# 🧬 ACFD-GAN – Faithful Re-implementation (Single-Class Mode)

**Paper:** *ACFD-GAN: Adaptive Conditional Feature Disentanglement GAN for Diabetic Retinopathy Augmentation* (Biomedical Signal Processing and Control, 2026)

**Pipeline theo paper:**


**Metrics:** FID (per epoch), WMA-FID, MSE (per grade), G/D losses

**Platform:** Kaggle GPU (T4 / P100)


In [ ]:
# Install required packages
import subprocess, sys
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "opencv-python-headless", "tqdm",
    "torchmetrics[image]", "torch-fidelity", "scipy", "scikit-image"
], check=True)
print("Packages ready.")

In [ ]:
import os, random, warnings, time, math, shutil, glob
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm import tqdm
import contextlib
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.utils as vutils
import torchvision.models as tvmodels
try:
    from skimage.filters import frangi
except Exception:
    frangi = None

warnings.filterwarnings("ignore")
print("PyTorch:", torch.__version__)
print("CUDA   :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU    :", torch.cuda.get_device_name(0))

GRADE_NAMES = {0: "No DR", 1: "Mild", 2: "Moderate", 3: "Severe", 4: "Proliferative"}
COLORS = ["#2ecc71", "#3498db", "#f39c12", "#e74c3c", "#9b59b6"]


## 1. Configuration


In [ ]:
class Config:
    # ── Paths (Kaggle) ─────────────────────────────────────────────────
    DATA_DIR   = "/kaggle/input/competitions/aptos2019-blindness-detection"
    TRAIN_CSV  = f"{DATA_DIR}/train.csv"
    IMG_DIR    = f"{DATA_DIR}/train_images"
    OUTPUT_DIR = "/kaggle/working/acfdgan_generated"
    CKPT_DIR   = "/kaggle/working/acfdgan_checkpoints"
    LOG_DIR    = "/kaggle/working"

    # ── Image ──────────────────────────────────────────────────────────
    IMG_SIZE     = 256        # resize full fundus to 256x256 for this practical baseline
    CROP_SOURCE_SIZE = 256    # 256 disables crop; crop-512 performed worse without CAR-UNet masks
    CROP_MIN_FUNDUS_FRAC = 0.60 # only used if CROP_SOURCE_SIZE > IMG_SIZE
    IMG_CHANNELS = 3
    MASK_CHANNELS = 4       # semantic pseudo-label channels: vessel, bright, dark, optic disc

    # ── CLAHE preprocessing (paper params) ────────────────────────────
    CLAHE_CLIP = 2.0
    CLAHE_TILE = (8, 8)

    # ── WAE (Wasserstein AutoEncoder) ──────────────────────────────────
    WAE_LATENT_DIM   = 128    # paper latent vector size
    WAE_EPOCHS       = 200    # paper: 200 epochs
    WAE_BATCH_SIZE   = 16
    WAE_LR           = 1e-4
    WAE_LAMBDA_MMD   = 10.0   # MMD regularisation weight
    WAE_SAVE_EVERY   = None   # do not save periodic WAE checkpoints on Kaggle

    # ── GAN Training ───────────────────────────────────────────────────
    GAN_EPOCHS       = 400    # stable pre-EMA branch that reached FID ~21.x
    GAN_BATCH_SIZE   = 8      # U-Net 256 + perceptual loss: memory heavy
    GAN_LR_G         = 2e-4   # paper: Adam(0.0002, 0.5, 0.999)
    GAN_LR_D         = 2e-4   # restore the stable best branch: keep D learning with G
    GAN_BETA1        = 0.5
    GAN_BETA2        = 0.999
    ADV_LOSS_TYPE    = "bce"  # BCE is the current best branch for Grade 3
    LAMBDA_L1        = 10.0   # paper weight; try 10/5 after baseline
    LAMBDA_VGG       = 10.0   # best observed branch so far
    # (start_epoch, lr_G, lr_D): stable branch uses matched G/D decay.
    GAN_LR_DECAY_EPOCHS = [(101, 1e-4, 1e-4), (201, 5e-5, 5e-5), (301, 2e-5, 2e-5)]
    NOISE_DIM        = 128    # concat with WAE latent

    # ── WMA-FID model selection (paper: window 5 or 7) ────────────────
    WMA_WINDOW       = 5

    # ──────────────────────────────────────────────────────────────────
    # SINGLE-CLASS MODE: set 0-4, or None for all grades
    # Paper trains each grade separately for 100 epochs
    # ──────────────────────────────────────────────────────────────────
    TARGET_CLASS     = 3

    # ── Generation ─────────────────────────────────────────────────────
    TARGET_PER_CLASS    = 1000
    MAX_GENERATE_PER_CLASS = None   # None = generate all needed
    GEN_BATCH           = 16

    # ── Runtime ────────────────────────────────────────────────────────
    RESUME            = False  # new Frangi + multi-layer VGG run; train fresh
    SAVE_EVERY        = None   # no periodic GAN checkpoints; keep output small
    EVAL_FREQ         = 10
    LOG_EVERY         = 5
    SHOW_PREVIEW_EVERY= 10
    FID_SUBSET        = None    # None = use the full active class for less noisy FID
    FID_FEATURE       = 2048   # Standard InceptionV3 pool3 features for paper-comparable FID
    MAX_TRAIN_SECONDS = float("inf")  # Disable time guard so WAE/GAN can finish all epochs
    GEN_BATCH_FID     = 16
    CACHE_VGG_FEATURES = True  # deterministic resize-256 baseline
    AUGMENT_TRAIN     = False
    AUG_FLIP_P        = 0.5
    AUG_ROT90_P       = 0.25
    AUG_BRIGHTNESS    = 0.06
    AUG_CONTRAST      = 0.06
    GEN_CANDIDATES_PER_MASK = 2
    FID_CANDIDATES_PER_MASK = 2# standard FID; no best-of-K/cherry-picking
    FID_CANDIDATE_CHUNK = 1
    MASK_MODE = "softfusion_v2_rich"  # Richer soft-fusion v2: keeps weaker vessel/lesion cues.
    MASK_INNER_ERODE = 5                # less aggressive rim removal than 9px kernel
    FRANGI_SIGMA_MAX = 5                # inclusive: sigmas [1,2,3,4,5]
    HYBRID_CANNY_LOW = 30
    HYBRID_CANNY_HIGH = 90
    HYBRID_FRANGI_WEIGHT = 0.7
    POSTHOC_FID_SEEDS = [42, 123, 2026]
    GRAD_CLIP_G_MAX_NORM = 1.0  # tame occasional generator spikes without changing the objective
    USE_EMA          = False# baseline rollback: live G previously reached FID ~21.x
    EMA_DECAY        = 0.999  # kept for later ablations; disabled while USE_EMA=False
    EMA_START_EPOCH  = 1
    R1_GAMMA = 0.0               # disabled: R1 + slow D destabilized the latest run
    R1_INTERVAL = 16             # lazy R1: apply every N discriminator steps
    USE_LESION_INPAINT_GAN = True
    INPAINT_ALPHA_GAIN = 1.22
    INPAINT_ALPHA_BLUR = 19
    INPAINT_ALPHA_MAX = 0.80
    BACKGROUND_PRESERVE_WEIGHT = 2.0
    EXPERIMENT_SUFFIX = 'resize256_softfusion_v2_rich_lesion_inpaint_alpha122_blur19_k2'# baseline + lesion-region inpainting
    SAVE_BEST_FID     = True
    SAVE_BEST_MSE     = True
    SAVE_WMA_MODEL    = False
    SAVE_PREVIEWS     = False
    CLEAN_OLD_HEAVY_CKPTS = True
    PACKAGE_OUTPUT  = False   # do not create zip bundles on Kaggle; they duplicate output size

    SEED   = 42
    DEVICE = "cuda" if __import__("torch").cuda.is_available() else "cpu"


cfg = Config()

if cfg.TARGET_CLASS is not None:
    ACTIVE_CLASSES = [cfg.TARGET_CLASS]
    RUN_TAG = f"g{cfg.TARGET_CLASS}"
else:
    ACTIVE_CLASSES = list(range(5))
    RUN_TAG = "all"
aug_tag = "aug" if cfg.AUGMENT_TRAIN else "noaug"
RUN_TAG = (f"{RUN_TAG}_{cfg.EXPERIMENT_SUFFIX}_{cfg.MASK_MODE}_{cfg.ADV_LOSS_TYPE}_"
           f"l1{cfg.LAMBDA_L1:g}_vgg{cfg.LAMBDA_VGG:g}_{aug_tag}_"
           f"cand{cfg.GEN_CANDIDATES_PER_MASK}_fidstandard")

cfg.OUTPUT_DIR = f"{cfg.OUTPUT_DIR}_{RUN_TAG}"
cfg.CKPT_DIR   = f"{cfg.CKPT_DIR}_{RUN_TAG}"
cfg.AUG_CSV    = f"{cfg.LOG_DIR}/train_augmented_acfdgan_{RUN_TAG}.csv"
cfg.WAE_CKPT   = f"{cfg.CKPT_DIR}/wae_best.pt"

for d in [cfg.OUTPUT_DIR, cfg.CKPT_DIR]:
    os.makedirs(d, exist_ok=True)

if cfg.CLEAN_OLD_HEAVY_CKPTS:
    cleaned = 0
    for pattern in [
        "/kaggle/working/acfdgan_checkpoints*/gan_epoch*.pt",
        "/kaggle/working/acfdgan_checkpoints*/wae_epoch*.pt",
        "/kaggle/working/acfdgan_checkpoints*/preview_ep*.png",
        "/kaggle/working/acfdgan_checkpoints*/best_generator_wmafid.pt",
        "/kaggle/working/acfdgan_checkpoints*/wae_best.pt",
        "/kaggle/working/acfdgan_images*.zip",
        "/kaggle/working/acfdgan_checkpoints*.zip",
        "/kaggle/working/acfdgan_full_bundle*.zip",
    ]:
        for p in glob.glob(pattern):
            try:
                os.remove(p); cleaned += 1
            except OSError:
                pass
    if cleaned:
        print(f"[CLEANUP] Removed {cleaned} old heavy checkpoint/preview files.")

print("=" * 60)
print(f"  Model           : ACFD-GAN (paper re-implementation)")
print(f"  Device          : {cfg.DEVICE}")
print(f"  Image size      : {cfg.IMG_SIZE}x{cfg.IMG_SIZE}")
print(f"  WAE latent dim  : {cfg.WAE_LATENT_DIM}")
print(f"  WAE epochs      : {cfg.WAE_EPOCHS}")
print(f"  GAN epochs      : {cfg.GAN_EPOCHS}")
print(f"  Adv loss        : {cfg.ADV_LOSS_TYPE}")
print(f"  Loss weights    : L1={cfg.LAMBDA_L1}, VGG={cfg.LAMBDA_VGG}")
print(f"  WMA window      : {cfg.WMA_WINDOW}")
print(f"  EMA enabled     : {cfg.USE_EMA}  (decay={cfg.EMA_DECAY})")
print(f"  Run tag         : {RUN_TAG}")
if cfg.TARGET_CLASS is not None:
    print(f"  *** SINGLE-CLASS: Grade {cfg.TARGET_CLASS} ({GRADE_NAMES[cfg.TARGET_CLASS]}) ***")


## 2. Seed & Utility Functions


In [ ]:
def set_seed(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True

def apply_clahe_rgb(img_bgr, clip=2.0, tile=(8,8)):
    """Apply CLAHE to each RGB channel independently, return RGB uint8.
    Restored to original per-channel RGB CLAHE (Plan 2 — EMA only run).
    LAB CLAHE caused FID=290 vs RGB CLAHE FID=25, so reverted.
    """
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    clahe   = cv2.createCLAHE(clipLimit=clip, tileGridSize=tile)
    out     = np.stack([clahe.apply(img_rgb[:, :, c]) for c in range(3)], axis=2)
    return out  # HxWx3 uint8 RGB

def _norm01(x, pct=99.5):
    x = np.nan_to_num(x).astype(np.float32)
    hi = np.percentile(x, pct)
    return np.clip(x / (hi + 1e-8), 0, 1)

def _multi_tophat(x):
    vals = []
    for k, w in ((7, 0.45), (11, 0.35), (17, 0.20)):
        resp = cv2.morphologyEx(x, cv2.MORPH_TOPHAT, np.ones((k,k), np.uint8))
        vals.append(w * _norm01(resp, pct=99.0))
    return np.clip(sum(vals), 0, 1)

def _multi_blackhat(x):
    vals = []
    for k, w in ((7, 0.45), (11, 0.35), (17, 0.20)):
        resp = cv2.morphologyEx(x, cv2.MORPH_BLACKHAT, np.ones((k,k), np.uint8))
        vals.append(w * _norm01(resp, pct=99.0))
    return np.clip(sum(vals), 0, 1)

def _vessel_continuity(frangi_v):
    seed = (frangi_v > 0.18).astype(np.uint8)
    seed = cv2.morphologyEx(seed, cv2.MORPH_CLOSE, np.ones((3,3), np.uint8))
    seed = cv2.morphologyEx(seed, cv2.MORPH_OPEN, np.ones((2,2), np.uint8))
    return cv2.GaussianBlur(seed.astype(np.float32), (5,5), 0)

def _optic_disc_prob(hsv):
    od_seed = ((hsv[:,:,2] > 205) & (hsv[:,:,1] < 95)).astype(np.uint8)
    num, labels, stats, _ = cv2.connectedComponentsWithStats(od_seed, connectivity=8)
    od = np.zeros_like(od_seed, dtype=np.float32)
    if num > 1:
        largest = 1 + np.argmax(stats[1:, cv2.CC_STAT_AREA])
        if stats[largest, cv2.CC_STAT_AREA] > 20:
            od_bin = (labels == largest).astype(np.uint8)
            od_bin = cv2.dilate(od_bin, np.ones((9,9), np.uint8), iterations=1)
            dist = cv2.distanceTransform(od_bin, cv2.DIST_L2, 5)
            od = _norm01(dist, pct=100.0)
            od = cv2.GaussianBlur(od, (11,11), 0)
    return np.clip(od, 0, 1)

def make_weak_mask(img_bgr, size=256):
    """softfusion_v2_rich: multi-scale soft pseudo-labels that retain weaker vessel and lesion cues."""
    if frangi is None:
        raise ImportError("softfusion_v2_rich requires scikit-image Frangi support.")
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    resized = cv2.resize(gray, (size, size), interpolation=cv2.INTER_AREA)
    img_s = cv2.resize(img_rgb, (size, size), interpolation=cv2.INTER_AREA)
    hsv = cv2.cvtColor(img_s, cv2.COLOR_RGB2HSV)
    green = img_s[:,:,1].astype(np.float32)

    fundus = (resized > 8).astype(np.uint8)
    k = max(3, int(getattr(cfg, "MASK_INNER_ERODE", 5)))
    if k % 2 == 0: k += 1
    inner = cv2.erode(fundus, np.ones((k,k), np.uint8), iterations=1).astype(np.float32)

    gray_f = resized.astype(np.float32) / 255.0
    sigma_max = int(getattr(cfg, "FRANGI_SIGMA_MAX", 5))
    frangi_v = cv2.GaussianBlur(_norm01(frangi(gray_f, sigmas=range(1, sigma_max + 1), black_ridges=True)), (3,3), 0)
    local_mean = cv2.blur(green, (17,17))
    deficit = _norm01(np.clip(local_mean - green, 0, None), pct=99.0)
    continuity = _vessel_continuity(frangi_v)
    blur = cv2.GaussianBlur(resized, (5,5), 0)
    canny = cv2.GaussianBlur((cv2.Canny(blur, 30, 90).astype(np.float32) / 255.0), (3,3), 0)
    vessel = np.clip(0.52 * frangi_v + 0.18 * continuity + 0.20 * deficit + 0.10 * canny, 0, 1) * inner

    v = hsv[:,:,2].astype(np.float32)
    s = hsv[:,:,1].astype(np.float32)
    bright_multi = _multi_tophat(v)
    bright_prior = np.clip((v - 160.0) / 95.0, 0, 1) * np.clip((130.0 - s) / 130.0, 0, 1)
    bright_value = _norm01(v, pct=99.5)
    bright = cv2.GaussianBlur(np.clip(0.58 * bright_multi + 0.27 * bright_prior + 0.15 * bright_value, 0, 1), (5,5), 0)

    dark_multi = _multi_blackhat(green)
    red_minus_green = _norm01(np.clip(img_s[:,:,0].astype(np.float32) - green, 0, None), pct=99.0)
    dark = cv2.GaussianBlur(np.clip(0.45 * deficit + 0.40 * dark_multi + 0.15 * red_minus_green, 0, 1), (5,5), 0)

    od = _optic_disc_prob(hsv)
    bright = bright * (1.0 - od) * inner
    dark = dark * inner
    od = od * inner
    return np.clip(np.stack([vessel, bright, dark, od], axis=2).astype(np.float32), 0, 1)

def choose_crop_top_left(support, crop_size=256, random_crop=True,
                         min_fundus_frac=0.60, max_tries=32):
    """Choose a 256 crop from a 512 support mask while avoiding mostly-black crops."""
    h, w = support.shape
    if h <= crop_size or w <= crop_size:
        return 0, 0
    max_top, max_left = h - crop_size, w - crop_size
    center_top, center_left = max_top // 2, max_left // 2
    if not random_crop:
        return center_top, center_left

    best = (center_top, center_left)
    best_frac = -1.0
    for _ in range(max_tries):
        # Mostly sample near the fundus center, sometimes sample uniformly for diversity.
        if np.random.rand() < 0.75:
            top = int(np.clip(np.random.normal(center_top, max_top / 5), 0, max_top))
            left = int(np.clip(np.random.normal(center_left, max_left / 5), 0, max_left))
        else:
            top = np.random.randint(0, max_top + 1)
            left = np.random.randint(0, max_left + 1)
        frac = support[top:top+crop_size, left:left+crop_size].mean()
        if frac > best_frac:
            best = (top, left)
            best_frac = frac
        if frac >= min_fundus_frac:
            return top, left
    return best

def preprocess_bgr_to_crop_tensors(img_bgr, img_size=256, crop_source_size=256,
                                   clahe_clip=2.0, clahe_tile=(8,8), random_crop=True,
                                   min_fundus_frac=0.60):
    """Resize to 256 by default; optionally resize larger then crop if configured."""
    if img_bgr is None:
        img_bgr = np.zeros((crop_source_size, crop_source_size, 3), np.uint8)
    src_size = max(crop_source_size, img_size)
    img_bgr = cv2.resize(img_bgr, (src_size, src_size), interpolation=cv2.INTER_AREA)
    rgb_full = apply_clahe_rgb(img_bgr, clahe_clip, clahe_tile)
    mask_full = make_weak_mask(img_bgr, src_size)

    gray_full = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    support = (gray_full > 8).astype(np.float32)
    top, left = choose_crop_top_left(support, img_size, random_crop, min_fundus_frac)

    rgb = rgb_full[top:top+img_size, left:left+img_size]
    mask = mask_full[top:top+img_size, left:left+img_size]
    img_f = (rgb.astype(np.float32) / 127.5) - 1.0
    mask_f = mask.astype(np.float32) * 2.0 - 1.0
    return (torch.from_numpy(img_f.transpose(2,0,1)),
            torch.from_numpy(mask_f.transpose(2,0,1)))

def safe_torch_load(path, device, label="checkpoint"):
    """Load a checkpoint, returning None if it is missing or corrupted."""
    path = Path(path)
    if not path.exists():
        return None
    try:
        return torch.load(path, map_location=device, weights_only=False)
    except Exception as e:
        print(f"[WARN] Cannot load {label} '{path.name}': {type(e).__name__}: {e}")
        print("       This usually means the file was interrupted while saving. It will be skipped.")
        return None

def tensor_to_bgr(tensor):
    """(C,H,W) in [-1,1] -> BGR uint8 numpy."""
    img = (tensor.cpu().detach().float() + 1.0) / 2.0
    img = (img * 255).clamp(0, 255).byte().permute(1, 2, 0).numpy()
    return cv2.cvtColor(img, cv2.COLOR_RGB2BGR)

set_seed(cfg.SEED)
print("Seed set:", cfg.SEED)

# CUDA sanity check
if cfg.DEVICE == "cuda":
    try:
        _t = torch.zeros(4, device="cuda"); _t.normal_(); del _t
        torch.cuda.empty_cache()
        print(f"CUDA OK | {torch.cuda.get_device_name(0)} | "
              f"sm_{torch.cuda.get_device_capability()[0]}{torch.cuda.get_device_capability()[1]}")
    except Exception as e:
        print(f"CUDA FAIL: {e} -> fallback CPU"); cfg.DEVICE = "cpu"


## 3. Dataset with Weak Mask

Paper: structure masks from CAR-UNet. Here we use OpenCV weak masks as substitute.


In [ ]:
class APTOSMaskDataset(Dataset):
    """
    Returns (img_tensor, mask_tensor, label):
      img_tensor : (3, 256, 256) float32 in [-1, 1]  — CLAHE-preprocessed RGB crop
      mask_tensor: (cfg.MASK_CHANNELS, 256, 256) float32 in [-1, 1]  — semantic pseudo-mask crop
      label      : int
    Caches deterministic 256x256 tensors by default. If crop_source_size > img_size,
    it can sample fundus-aware crops, but the practical baseline keeps this disabled.
    """
    def __init__(self, df, img_dir, img_size=256, crop_source_size=512,
                 min_fundus_frac=0.60, clahe_clip=2.0, clahe_tile=(8,8)):
        self.df = df.reset_index(drop=True)
        self.size = img_size
        self.use_vgg_cache = False
        self.cache_size = max(crop_source_size, img_size)
        self.min_fundus_frac = min_fundus_frac
        print(f"Caching {len(self.df)} images + masks at {self.cache_size}x{self.cache_size} into RAM...")
        self.imgs, self.masks, self.supports, self.labels = [], [], [], []
        for _, row in tqdm(self.df.iterrows(), total=len(self.df),
                           desc="  Cache", leave=False):
            path = os.path.join(img_dir, f"{row['id_code']}.png")
            img_bgr = cv2.imread(path)
            if img_bgr is None:
                img_bgr = np.zeros((self.cache_size, self.cache_size, 3), np.uint8)
            img_bgr = cv2.resize(img_bgr, (self.cache_size, self.cache_size),
                                 interpolation=cv2.INTER_AREA)
            support = (cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY) > 8).astype(np.float32)
            # CLAHE RGB
            rgb = apply_clahe_rgb(img_bgr, clahe_clip, clahe_tile)
            rgb_f = (rgb.astype(np.float32) / 127.5) - 1.0  # [-1,1]
            t_img = torch.from_numpy(rgb_f.transpose(2,0,1)).half()
            # Weak mask
            mask = make_weak_mask(img_bgr, self.cache_size)   # [0,1]
            mask_f = mask * 2.0 - 1.0                          # [-1,1]
            t_mask = torch.from_numpy(mask_f.transpose(2,0,1)).half()
            self.imgs.append(t_img)
            self.masks.append(t_mask)
            self.supports.append(torch.from_numpy(support).half())
            self.labels.append(int(row["diagnosis"]))
        self.imgs  = torch.stack(self.imgs)
        self.masks = torch.stack(self.masks)
        self.supports = torch.stack(self.supports)
        self.labels= torch.tensor(self.labels, dtype=torch.long)
        print(f"  Ready: {len(self.labels)} samples, "
              f"img={tuple(self.imgs.shape)}, mask={tuple(self.masks.shape)}")

    def _random_crop_pair(self, img, mask, support=None):
        if self.cache_size == self.size:
            return img, mask
        if support is None:
            support = (img.mean(0) > -0.95).float()
        top, left = choose_crop_top_left(
            support.float().cpu().numpy(), self.size, True, self.min_fundus_frac)
        return (img[:, top:top+self.size, left:left+self.size],
                mask[:, top:top+self.size, left:left+self.size])

    def __len__(self): return len(self.labels)
    def _augment_pair(self, img, mask):
        if not getattr(self, "augment_train", False):
            return img, mask
        if torch.rand(()) < getattr(self, "aug_flip_p", 0.0):
            img = torch.flip(img, dims=[2]); mask = torch.flip(mask, dims=[2])
        if torch.rand(()) < getattr(self, "aug_flip_p", 0.0):
            img = torch.flip(img, dims=[1]); mask = torch.flip(mask, dims=[1])
        if torch.rand(()) < getattr(self, "aug_rot90_p", 0.0):
            k = int(torch.randint(0, 4, (1,)).item())
            img = torch.rot90(img, k, dims=[1,2]); mask = torch.rot90(mask, k, dims=[1,2])
        b = getattr(self, "aug_brightness", 0.0)
        c = getattr(self, "aug_contrast", 0.0)
        if b > 0:
            img = img + (torch.rand(()) * 2 - 1) * b
        if c > 0:
            mean = img.mean(dim=(1,2), keepdim=True)
            factor = 1.0 + (torch.rand(()) * 2 - 1) * c
            img = (img - mean) * factor + mean
        return img.clamp(-1, 1), mask.clamp(-1, 1)

    def __getitem__(self, i):
        img, mask = self._random_crop_pair(
            self.imgs[i].float(), self.masks[i].float(), self.supports[i].float())
        img, mask = self._augment_pair(img, mask)
        if self.use_vgg_cache and hasattr(self, "vgg_feats"):
            feats = tuple(f[i].float() for f in self.vgg_feats)
            return (img, mask, self.labels[i], feats)
        return (img, mask, self.labels[i])


df_full = pd.read_csv(cfg.TRAIN_CSV)
if cfg.TARGET_CLASS is not None:
    df = df_full[df_full["diagnosis"] == cfg.TARGET_CLASS].reset_index(drop=True)
    print(f"[Single-Class] Grade {cfg.TARGET_CLASS} ({GRADE_NAMES[cfg.TARGET_CLASS]}): {len(df)} images")
else:
    df = df_full.copy()
    print(f"[Multi-Class] Total: {len(df)} images")

# EDA
counts_full = df_full["diagnosis"].value_counts().sort_index()
print(f"\n{'Grade':<8}{'Name':<16}{'Count':>6}{'Need Gen':>10}")
print("-"*45)
for g, cnt in counts_full.items():
    need = max(0, cfg.TARGET_PER_CLASS - cnt)
    print(f"  {g:<6}{GRADE_NAMES[g]:<16}{cnt:>6}{need:>10,}")

dataset    = APTOSMaskDataset(df, cfg.IMG_DIR, cfg.IMG_SIZE, cfg.CROP_SOURCE_SIZE,
                              cfg.CROP_MIN_FUNDUS_FRAC, cfg.CLAHE_CLIP, cfg.CLAHE_TILE)
dataset.augment_train = cfg.AUGMENT_TRAIN
dataset.aug_flip_p = cfg.AUG_FLIP_P
dataset.aug_rot90_p = cfg.AUG_ROT90_P
dataset.aug_brightness = cfg.AUG_BRIGHTNESS
dataset.aug_contrast = cfg.AUG_CONTRAST
dataloader = DataLoader(dataset, batch_size=cfg.GAN_BATCH_SIZE,
                        shuffle=True, num_workers=0,
                        pin_memory=True, drop_last=True)
print(f"Steps/epoch: {len(dataloader)}")


## 4. WAE (Wasserstein AutoEncoder) Architecture

Paper: WAE-MMD trained 200 epochs → encoder produces latent z → used to condition ACFD-GAN generator


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# WAE Architecture
# Encoder: 4x Conv(3x3, stride2) + BN + LeakyReLU → FC → latent z
# Decoder: FC → reshape → 4x ConvTranspose → Tanh
# Trained with reconstruction loss (MSE) + MMD regularisation
# ─────────────────────────────────────────────────────────────────────────────

class WAEEncoder(nn.Module):
    def __init__(self, img_channels=3, latent_dim=128, img_size=256):
        super().__init__()
        # 4 stride-2 convs: 256->128->64->32->16
        self.conv = nn.Sequential(
            nn.Conv2d(img_channels, 64,  4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, True),
            nn.Conv2d(64,  128, 4, 2, 1, bias=False), nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2, True),
            nn.Conv2d(128, 256, 4, 2, 1, bias=False), nn.BatchNorm2d(256),
            nn.LeakyReLU(0.2, True),
            nn.Conv2d(256, 512, 4, 2, 1, bias=False), nn.BatchNorm2d(512),
            nn.LeakyReLU(0.2, True),
        )  # -> (B, 512, 16, 16)
        flat = 512 * (img_size // 16) * (img_size // 16)
        self.fc = nn.Linear(flat, latent_dim)
        self._flat = flat

    def forward(self, x):
        h = self.conv(x)              # (B,512,16,16)
        h = h.view(h.size(0), -1)    # (B, flat)
        return self.fc(h)             # (B, latent_dim)


class WAEDecoder(nn.Module):
    def __init__(self, img_channels=3, latent_dim=128, img_size=256):
        super().__init__()
        self._s = img_size // 16      # = 16
        flat = 512 * self._s * self._s
        self.fc = nn.Sequential(
            nn.Linear(latent_dim, flat), nn.ReLU(True)
        )
        self.deconv = nn.Sequential(
            nn.ConvTranspose2d(512, 256, 4, 2, 1, bias=False),
            nn.BatchNorm2d(256), nn.ReLU(True),
            nn.ConvTranspose2d(256, 128, 4, 2, 1, bias=False),
            nn.BatchNorm2d(128), nn.ReLU(True),
            nn.ConvTranspose2d(128, 64,  4, 2, 1, bias=False),
            nn.BatchNorm2d(64),  nn.ReLU(True),
            nn.ConvTranspose2d(64, img_channels, 4, 2, 1, bias=False),
            nn.Tanh(),
        )

    def forward(self, z):
        h = self.fc(z).view(z.size(0), 512, self._s, self._s)
        return self.deconv(h)


class WAE(nn.Module):
    def __init__(self, img_channels=3, latent_dim=128, img_size=256):
        super().__init__()
        self.encoder = WAEEncoder(img_channels, latent_dim, img_size)
        self.decoder = WAEDecoder(img_channels, latent_dim, img_size)

    def encode(self, x): return self.encoder(x)
    def decode(self, z): return self.decoder(z)
    def forward(self, x):
        z = self.encode(x)
        return self.decode(z), z


def mmd_loss(z, reg_weight=10.0, sigmas=(1.0, 2.0, 4.0, 8.0, 16.0)):
    """Unbiased multi-scale RBF MMD between z and N(0,1)."""
    z = z.float()  # Prevent float16 overflow in AMP
    B = z.size(0)
    z_prior = torch.randn_like(z)

    def pdist2(x, y):
        xx = (x**2).sum(1, keepdim=True)
        yy = (y**2).sum(1, keepdim=True)
        return (xx + yy.t() - 2 * (x @ y.t())).clamp(min=0)

    d_zz = pdist2(z, z)
    d_pp = pdist2(z_prior, z_prior)
    d_zp = pdist2(z, z_prior)
    eye = torch.eye(B, device=z.device, dtype=torch.bool)

    mmd = 0.0
    for sigma in sigmas:
        k_zz = torch.exp(-d_zz / (2 * sigma**2)).masked_fill(eye, 0).sum() / max(1, B * (B - 1))
        k_pp = torch.exp(-d_pp / (2 * sigma**2)).masked_fill(eye, 0).sum() / max(1, B * (B - 1))
        k_zp = torch.exp(-d_zp / (2 * sigma**2)).mean()
        mmd = mmd + k_zz + k_pp - 2 * k_zp
    return reg_weight * mmd / len(sigmas)


# Instantiate WAE
wae = WAE(cfg.IMG_CHANNELS, cfg.WAE_LATENT_DIM, cfg.IMG_SIZE).to(cfg.DEVICE)
n_wae = sum(p.numel() for p in wae.parameters() if p.requires_grad)
print(f"WAE params: {n_wae:,}")


## 5. WAE Training (200 epochs)

Paper: pretrain WAE on full training set for 200 epochs, then freeze encoder.


In [ ]:
wae_ckpt_path = Path(cfg.CKPT_DIR) / "wae_best.pt"
wae_recon_ckpt_path = Path(cfg.CKPT_DIR) / "wae_best_recon.pt"
wae_losses, wae_recon_losses, wae_mmd_losses = [], [], []

# DataLoader for WAE — use only image tensors (not mask)
wae_loader = DataLoader(dataset, batch_size=cfg.WAE_BATCH_SIZE,
                        shuffle=True, num_workers=0,
                        pin_memory=True, drop_last=True)

opt_wae = optim.Adam(wae.parameters(), lr=cfg.WAE_LR, betas=(0.9, 0.999))

# AMP
use_amp = (cfg.DEVICE == "cuda")
try:
    from torch.amp import GradScaler as _GS
    scaler_wae = _GS(device=cfg.DEVICE, enabled=use_amp)
except:
    from torch.cuda.amp import GradScaler as _GS
    scaler_wae = _GS(enabled=use_amp)

@contextlib.contextmanager
def autocast(enabled=True):
    if not enabled: yield; return
    try:
        with torch.autocast(device_type=cfg.DEVICE, enabled=True): yield
    except TypeError:
        from torch.cuda.amp import autocast as _ac
        with _ac(enabled=True): yield

wae_start_epoch = 1
best_wae_loss   = float("inf")
best_wae_recon  = float("inf")

resume_wae_path = wae_recon_ckpt_path if wae_recon_ckpt_path.exists() else wae_ckpt_path
wae_ck = safe_torch_load(resume_wae_path, cfg.DEVICE, "WAE resume") if cfg.RESUME else None
if cfg.RESUME and wae_ck is not None:
    wae.load_state_dict(wae_ck["wae_state"])
    if "optimizer" in wae_ck:
        opt_wae.load_state_dict(wae_ck["optimizer"])
    wae_start_epoch = wae_ck.get("epoch", 0) + 1
    best_wae_loss   = wae_ck.get("best_loss", float("inf"))
    best_wae_recon  = wae_ck.get("best_recon", float("inf"))
    wae_losses      = wae_ck.get("losses", [])
    wae_recon_losses= wae_ck.get("recon_losses", [])
    wae_mmd_losses  = wae_ck.get("mmd_losses", [])
    print(f"[WAE RESUME] {resume_wae_path.name} | epoch={wae_start_epoch}, "
          f"best_loss={best_wae_loss:.4f}, best_recon={best_wae_recon:.6f}")
else:
    print("[WAE FRESH] Training from scratch.")

wae_wall_start = time.perf_counter()
print(f"Training WAE for {cfg.WAE_EPOCHS} epochs, "
      f"{len(wae_loader)} steps/epoch")
print("=" * 55)

for ep in range(wae_start_epoch, cfg.WAE_EPOCHS + 1):
    wae.train()
    ep_loss = ep_recon = ep_mmd = 0.0
    for imgs, masks, labels in wae_loader:
        imgs = imgs.to(cfg.DEVICE, non_blocking=True)
        opt_wae.zero_grad(set_to_none=True)
        with autocast(use_amp):
            recon, z = wae(imgs)
            loss_recon = F.mse_loss(recon, imgs)
            loss_mmd   = mmd_loss(z, cfg.WAE_LAMBDA_MMD)
            loss_total = loss_recon + loss_mmd
        scaler_wae.scale(loss_total).backward()
        scaler_wae.step(opt_wae)
        scaler_wae.update()
        ep_loss  += loss_total.item()
        ep_recon += loss_recon.item()
        ep_mmd   += loss_mmd.item()
    ep_loss  /= len(wae_loader)
    ep_recon /= len(wae_loader)
    ep_mmd   /= len(wae_loader)
    wae_losses.append(ep_loss)
    wae_recon_losses.append(ep_recon)
    wae_mmd_losses.append(ep_mmd)

    if ep % cfg.LOG_EVERY == 0 or ep == cfg.WAE_EPOCHS:
        print(f"  WAE Epoch {ep:3d}/{cfg.WAE_EPOCHS} | loss={ep_loss:.4f} | "
              f"recon={ep_recon:.6f} | mmd={ep_mmd:.4f}")

    if ep_loss < best_wae_loss:
        best_wae_loss = ep_loss

    if ep_recon < best_wae_recon:
        best_wae_recon = ep_recon
        torch.save({"epoch":ep, "wae_state":wae.state_dict(),
                    "best_loss":best_wae_loss,
                    "best_recon":best_wae_recon}, wae_recon_ckpt_path)

    if cfg.WAE_SAVE_EVERY and ep % cfg.WAE_SAVE_EVERY == 0:
        torch.save({"epoch":ep, "wae_state":wae.state_dict(),
                    "best_loss":best_wae_loss,
                    "best_recon":best_wae_recon},
                   Path(cfg.CKPT_DIR) / f"wae_epoch{ep:04d}.pt")

    if (time.perf_counter() - wae_wall_start) > cfg.MAX_TRAIN_SECONDS * 0.35:
        print(f"[TIME GUARD] WAE stopped at epoch {ep}"); break

# Load reconstruction-selected WAE and freeze encoder.
# The GAN needs useful image latents more than a low total MMD-dominated WAE loss.
load_wae_path = wae_recon_ckpt_path if wae_recon_ckpt_path.exists() else wae_ckpt_path
load_wae_ck = safe_torch_load(load_wae_path, cfg.DEVICE, "selected WAE")
if load_wae_ck is None and load_wae_path != wae_ckpt_path:
    load_wae_path = wae_ckpt_path
    load_wae_ck = safe_torch_load(load_wae_path, cfg.DEVICE, "fallback WAE")
if load_wae_ck is not None:
    wae.load_state_dict(load_wae_ck["wae_state"])
else:
    print("[WARN] No valid WAE checkpoint found after training; keeping current WAE weights.")
wae.eval()
for p in wae.parameters(): p.requires_grad_(False)
print(f"\nWAE done. Loaded {load_wae_path.name} | "
      f"best_loss={best_wae_loss:.4f} | best_recon={best_wae_recon:.6f} | Encoder frozen.")

# Plot WAE loss
if len(wae_losses) > 0:
    plt.figure(figsize=(9,3))
    plt.plot(wae_losses, color="#3498db", lw=1.5, label="Recon + MMD")
    if len(wae_recon_losses) == len(wae_losses):
        plt.plot(wae_recon_losses, color="#2ecc71", lw=1.5, label="Recon")
    plt.title("WAE Training Loss"); plt.xlabel("Epoch"); plt.ylabel("Loss")
    plt.legend()
    plt.grid(alpha=0.3); plt.tight_layout()
    plt.savefig(f"{cfg.LOG_DIR}/wae_loss_{RUN_TAG}.png", dpi=120)
    plt.show()


## 6. ACFD-GAN Architecture

**Generator (U-Net style):**
- Encoder: 4× LRDB (Local Residual Dense Block) + downsample Conv(4x4, s2) + BN
- Decoder: bilinear upsample + Conv(3x3) + BN; skip connections via ACFF
- Each decoder layer: AMM (Adaptive Mask Modulation) + AdaIN from WAE latent

**Discriminator:** 5-layer PatchGAN — 3 stride-2, 2 stride-1

**Losses:** Adversarial + 10·L1 + 10·VGG19 Perceptual


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# LRDB: Lightweight Residual Dense Block (paper encoder building block)
# 1x1 reduction + depthwise-separable additive dense features + residual fusion
# ─────────────────────────────────────────────────────────────────────────────
class DepthwiseSeparableConv(nn.Module):
    def __init__(self, ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(ch, ch, 3, 1, 1, groups=ch, bias=False),
            nn.Conv2d(ch, ch, 1, 1, 0, bias=False),
            nn.LeakyReLU(0.2, True),
        )

    def forward(self, x):
        return self.net(x)


class LRDB(nn.Module):
    """Paper-style LRDB: reduced-width additive dense path + local residual fusion."""
    def __init__(self, in_ch, growth=32):
        super().__init__()
        self.reduce = nn.Sequential(
            nn.Conv2d(in_ch, growth, 1, 1, 0, bias=False),
            nn.LeakyReLU(0.2, True),
        )
        self.d1 = DepthwiseSeparableConv(growth)
        self.d2 = DepthwiseSeparableConv(growth)
        self.d3 = DepthwiseSeparableConv(growth)
        self.fuse = nn.Conv2d(in_ch + 4 * growth, in_ch, 1, bias=False)

    def forward(self, x):
        f0 = self.reduce(x)
        f1 = self.d1(f0)
        f2 = self.d2(f0 + f1)
        f3 = self.d3(f0 + f1 + f2)
        fused = self.fuse(torch.cat([x, f0, f1, f2, f3], dim=1))
        return x + fused


# ─────────────────────────────────────────────────────────────────────────────
# ACFF: Adaptive Cross-scale Feature Fusion (skip connection fusion)
# Channel + spatial attention, closer to paper eq(6)-(8)
# ─────────────────────────────────────────────────────────────────────────────
class ACFF(nn.Module):
    def __init__(self, ch, r=8):
        super().__init__()
        mid = max(ch // r, 4)
        # Channel attention: avg+max pooled fused feature -> channel weights.
        self.ca = nn.Sequential(
            nn.Conv2d(ch * 2, mid, 1), nn.ReLU(True),
            nn.Conv2d(mid, ch, 1), nn.Sigmoid()
        )
        # Spatial attention: avg+max across channels -> spatial lesion/fundus map.
        self.sa = nn.Sequential(
            nn.Conv2d(2, 1, 3, 1, 1, bias=False), nn.Sigmoid()
        )
        self.out = nn.Sequential(
            nn.Conv2d(ch * 2, ch, 1, bias=False),
            nn.BatchNorm2d(ch), nn.ReLU(True)
        )

    def forward(self, enc_feat, dec_feat):
        # enc_feat, dec_feat: (B, ch, H, W)
        if dec_feat.shape[-2:] != enc_feat.shape[-2:]:
            dec_feat = F.interpolate(dec_feat, size=enc_feat.shape[-2:],
                                     mode="bilinear", align_corners=False)
        fr = enc_feat + dec_feat
        avg = F.adaptive_avg_pool2d(fr, 1)
        mx  = F.adaptive_max_pool2d(fr, 1)
        alpha = self.ca(torch.cat([avg, mx], dim=1))
        ca_feat = fr * alpha
        sa_in = torch.cat([ca_feat.mean(1, keepdim=True),
                           ca_feat.max(1, keepdim=True)[0]], dim=1)
        beta = self.sa(sa_in)
        weighted = ca_feat * (1.0 + beta)
        return self.out(torch.cat([dec_feat, weighted], dim=1))


# ─────────────────────────────────────────────────────────────────────────────
# AMM: Adaptive Modulation Module (paper-aligned mask-conditioned AdaIN)
# ─────────────────────────────────────────────────────────────────────────────
class AMM(nn.Module):
    def __init__(self, ch, latent_dim, mask_ch=3):
        super().__init__()
        hidden = max(ch, latent_dim)
        self.norm = nn.InstanceNorm2d(ch, affine=False)
        self.mask_enc = nn.Sequential(
            nn.Conv2d(mask_ch, max(ch // 4, 8), 3, 1, 1, bias=False), nn.ReLU(True),
            nn.Conv2d(max(ch // 4, 8), ch, 3, 1, 1, bias=False), nn.ReLU(True),
            nn.AdaptiveAvgPool2d(1),
        )
        self.mask_proj = nn.Linear(ch, latent_dim)
        # Paper: 3 FC layers, 2 ReLU activations, and LayerNorm.
        self.mapping = nn.Sequential(
            nn.Linear(latent_dim * 2, hidden), nn.ReLU(True),
            nn.Linear(hidden, hidden), nn.ReLU(True),
            nn.LayerNorm(hidden),
            nn.Linear(hidden, ch * 2),
        )

    def forward(self, feat, mask, z):
        # Structural/lesion mask updates the latent vector before AdaIN modulation.
        m = F.interpolate(mask, size=feat.shape[2:], mode="bilinear",
                          align_corners=False)
        m = self.mask_enc(m).flatten(1)
        mask_latent = self.mask_proj(m)
        fused = torch.cat([z, mask_latent], dim=1)
        gamma, beta = self.mapping(fused).chunk(2, dim=1)
        gamma = gamma.unsqueeze(2).unsqueeze(3)
        beta  = beta.unsqueeze(2).unsqueeze(3)
        # Residual scale parameterization keeps early training close to identity.
        return (1.0 + gamma) * self.norm(feat) + beta


# ─────────────────────────────────────────────────────────────────────────────
# ACFD-GAN Generator
# Input: (mask, z_concat) where z_concat = cat(wae_z, random_noise)
# Architecture: U-Net with LRDB encoder, ACFF skip, AMM+AdaIN decoder
# ─────────────────────────────────────────────────────────────────────────────
class ACFDGenerator(nn.Module):
    def __init__(self, mask_ch=3, img_ch=3, latent_dim=256,
                 base_ch=64, growth=32):
        super().__init__()
        self.latent_dim = latent_dim
        c = base_ch  # 64

        # ── Encoder ──────────────────────────────────────────────────────
        # Initial conv: mask -> c feature maps
        self.enc_in = nn.Sequential(
            nn.Conv2d(mask_ch, c, 3, 1, 1, bias=False),
            nn.BatchNorm2d(c), nn.LeakyReLU(0.2, True)
        )  # (B, c, 256, 256)

        # 4x (LRDB + down)
        def enc_block(in_ch, out_ch):
            return nn.Sequential(
                LRDB(in_ch, growth),
                nn.Conv2d(in_ch, out_ch, 4, 2, 1, bias=False),
                nn.BatchNorm2d(out_ch), nn.LeakyReLU(0.2, True)
            )

        self.enc1 = enc_block(c,     c*2)   # 256->128, c->2c
        self.enc2 = enc_block(c*2,   c*4)   # 128->64,  2c->4c
        self.enc3 = enc_block(c*4,   c*8)   # 64->32,   4c->8c
        self.enc4 = enc_block(c*8,   c*8)   # 32->16,   8c->8c

        # Bottleneck: fc from latent z -> 8c spatial feature
        s4 = 16  # spatial size at enc4 output
        self.z_proj = nn.Linear(latent_dim, c*8 * s4 * s4)
        self.z_norm = nn.BatchNorm1d(c*8 * s4 * s4)
        self._s4    = s4

        # ── Skip Fusion (ACFF) ────────────────────────────────────────────
        self.acff4 = ACFF(c*8)
        self.acff3 = ACFF(c*4)
        self.acff2 = ACFF(c*2)
        self.acff1 = ACFF(c)

        # ── Decoder ──────────────────────────────────────────────────────
        # Paper-style AMM performs mask-conditioned latent mapping + AdaIN together.
        self.amm4  = AMM(c*8, latent_dim, mask_ch)
        self.amm3  = AMM(c*4, latent_dim, mask_ch)
        self.amm2  = AMM(c*2, latent_dim, mask_ch)
        self.amm1  = AMM(c, latent_dim, mask_ch)

        def dec_block(in_ch, out_ch):
            return nn.Sequential(
                nn.Conv2d(in_ch, out_ch, 3, 1, 1, bias=False),
                nn.BatchNorm2d(out_ch), nn.ReLU(True)
            )

        self.dec4 = dec_block(c*8,  c*4)   # 16->32
        self.dec3 = dec_block(c*4,  c*2)   # 32->64
        self.dec2 = dec_block(c*2,  c)   # 64->128
        self.dec1 = dec_block(c,    c)     # 128->256

        self.out_conv = nn.Sequential(
            nn.Conv2d(c, img_ch, 3, 1, 1), nn.Tanh()
        )

    def _up(self, x):
        return F.interpolate(x, scale_factor=2, mode="bilinear",
                             align_corners=False)

    def forward(self, mask, z):
        # z: (B, latent_dim) = cat(wae_z, random_noise)

        # ── Encode ─────────────────────────────────────────────────────
        e0 = self.enc_in(mask)     # (B, c, 256, 256)
        e1 = self.enc1(e0)         # (B, 2c, 128, 128)
        e2 = self.enc2(e1)         # (B, 4c, 64, 64)
        e3 = self.enc3(e2)         # (B, 8c, 32, 32)
        e4 = self.enc4(e3)         # (B, 8c, 16, 16)

        # Latent injection at bottleneck
        zp = F.relu(self.z_norm(self.z_proj(z)))
        zp = zp.view(z.size(0), -1, self._s4, self._s4)  # (B,8c,16,16)
        h  = e4 + zp

        # ── Decode with ACFF + AMM + AdaIN ─────────────────────────────
        # Level 4: 16->32
        h  = self._up(h)
        h  = self.acff4(e3, h)
        h  = self.amm4(h, mask, z)
        h  = self.dec4(h)          # (B, 4c, 32, 32)

        # Level 3: 32->64
        h  = self._up(h)
        h  = self.acff3(e2, h)
        h  = self.amm3(h, mask, z)
        h  = self.dec3(h)          # (B, 2c, 64, 64)

        # Level 2: 64->128
        h  = self._up(h)
        h  = self.acff2(e1, h)
        h  = self.amm2(h, mask, z)
        h  = self.dec2(h)          # (B, c, 128, 128)

        # Level 1: 128->256
        h  = self._up(h)
        h  = self.acff1(e0, h)
        h  = self.amm1(h, mask, z)
        h  = self.dec1(h)          # (B, c, 256, 256)

        return self.out_conv(h)    # (B, 3, 256, 256)


# ─────────────────────────────────────────────────────────────────────────────
# ACFD-GAN Discriminator — 5-layer PatchGAN
# 3 stride-2 layers, 2 stride-1 layers (paper Table 2)
# ─────────────────────────────────────────────────────────────────────────────
class ACFDDiscriminator(nn.Module):
    def __init__(self, img_ch=3, mask_ch=3, base_ch=64):
        super().__init__()
        SN = nn.utils.spectral_norm
        in_ch = img_ch + mask_ch  # concat image + mask

        def d_block(ic, oc, stride, first=False):
            layers = [SN(nn.Conv2d(ic, oc, 4, stride, 1, bias=False))]
            if not first:
                layers.append(nn.BatchNorm2d(oc))
            layers.append(nn.LeakyReLU(0.2, True))
            return nn.Sequential(*layers)

        c = base_ch
        # 3 stride-2 + 2 stride-1 (paper)
        self.net = nn.Sequential(
            d_block(in_ch, c,   2, first=True),   # 256->128
            d_block(c,     c*2, 2),               # 128->64
            d_block(c*2,   c*4, 2),               # 64->32
            d_block(c*4,   c*8, 1),               # 32->32
            SN(nn.Conv2d(c*8, 1, 4, 1, 1, bias=False))  # patch output
        )

    def forward(self, img, mask):
        x = torch.cat([img, mask], dim=1)
        return self.net(x)


# ─────────────────────────────────────────────────────────────────────────────
# VGG19 Perceptual Loss
# ─────────────────────────────────────────────────────────────────────────────
class VGGPerceptualLoss(nn.Module):
    def __init__(self, device):
        super().__init__()
        vgg = tvmodels.vgg19(weights=tvmodels.VGG19_Weights.IMAGENET1K_V1)
        feats = list(vgg.features)
        # Multi-layer perceptual features: shallow color/edge + deeper lesion structure.
        # Plan 2: reverted to original weights (0.1, 0.3, 0.6) — deeper semantic
        # emphasis was the stable best branch. LAB+shallow-VGG caused FID=290.
        self.slice1 = nn.Sequential(*feats[:9]).to(device)    # relu2_2
        self.slice2 = nn.Sequential(*feats[9:18]).to(device)  # relu3_4
        self.slice3 = nn.Sequential(*feats[18:27]).to(device) # relu4_4
        self.layer_weights = (0.1, 0.3, 0.6)  # original best-branch weights
        for p in self.parameters():
            p.requires_grad_(False)
        self.eval()
        # ImageNet normalisation
        mean = torch.tensor([0.485,0.456,0.406], device=device).view(1,3,1,1)
        std  = torch.tensor([0.229,0.224,0.225], device=device).view(1,3,1,1)
        self.register_buffer("mean", mean)
        self.register_buffer("std",  std)

    def extract_features(self, x):
        h1 = self.slice1(x)
        h2 = self.slice2(h1)
        h3 = self.slice3(h2)
        return (h1, h2, h3)

    def features(self, x):
        # Compatibility helper for candidate-selection code; use deepest feature.
        return self.extract_features(x)[-1]

    def forward(self, fake, real=None, real_feat_cached=None):
        xn = (fake * 0.5 + 0.5 - self.mean) / self.std
        fake_feats = self.extract_features(xn)
        if real_feat_cached is not None:
            real_feats = real_feat_cached
        else:
            yn = (real * 0.5 + 0.5 - self.mean) / self.std
            real_feats = self.extract_features(yn)
        return sum(w * F.l1_loss(f, r) for w, f, r in zip(self.layer_weights, fake_feats, real_feats))

# Instantiate
LATENT_DIM = cfg.WAE_LATENT_DIM + cfg.NOISE_DIM  # 128+128=256

G = ACFDGenerator(
    mask_ch=cfg.MASK_CHANNELS, img_ch=cfg.IMG_CHANNELS,
    latent_dim=LATENT_DIM, base_ch=64, growth=32
).to(cfg.DEVICE)

D = ACFDDiscriminator(
    img_ch=cfg.IMG_CHANNELS, mask_ch=cfg.MASK_CHANNELS, base_ch=64
).to(cfg.DEVICE)

vgg_loss = VGGPerceptualLoss(cfg.DEVICE)

g_params = sum(p.numel() for p in G.parameters() if p.requires_grad)
d_params = sum(p.numel() for p in D.parameters() if p.requires_grad)
print(f"Generator     params: {g_params:,}")
print(f"Discriminator params: {d_params:,}")
print(f"Latent dim (WAE+noise): {LATENT_DIM}")
print(f"Loss: Adv + {cfg.LAMBDA_L1}*L1 + {cfg.LAMBDA_VGG}*VGG19 Perceptual")

# Weight init (DCGAN style)
def weights_init(m):
    cn = m.__class__.__name__
    if "Conv" in cn and hasattr(m, "weight_orig"):
        nn.init.normal_(m.weight_orig.data, 0.0, 0.02)
    elif "Conv" in cn and hasattr(m, "weight"):
        nn.init.normal_(m.weight.data, 0.0, 0.02)
    elif "BatchNorm" in cn and hasattr(m, "weight"):
        nn.init.normal_(m.weight.data, 1.0, 0.02)
        nn.init.constant_(m.bias.data, 0.0)

G.apply(weights_init)
D.apply(weights_init)


## 7. ACFD-GAN Training Loop

Loss = Adversarial + 10·L1 + 10·VGG19 Perceptual. Model selection: WMA-FID.


In [ ]:
# ── Optimisers ────────────────────────────────────────────────────────────────
optimizer_G = optim.Adam(G.parameters(),
                         lr=cfg.GAN_LR_G, betas=(cfg.GAN_BETA1, cfg.GAN_BETA2))
optimizer_D = optim.Adam(D.parameters(),
                         lr=cfg.GAN_LR_D, betas=(cfg.GAN_BETA1, cfg.GAN_BETA2))

if cfg.ADV_LOSS_TYPE.lower() == "bce":
    criterion_adv = nn.BCEWithLogitsLoss()  # Paper-style BCE on discriminator logits
elif cfg.ADV_LOSS_TYPE.lower() == "lsgan":
    criterion_adv = nn.MSELoss()            # Often smoother/lower-FID GAN ablation
else:
    raise ValueError(f"Unsupported ADV_LOSS_TYPE: {cfg.ADV_LOSS_TYPE}")

# ── Optional EMA Generator (disabled for the FID~21 baseline) ─────────────────
# Shadow copy of G with exponential moving average of weights.
# Keep the shadow copy available for future ablations, but the baseline uses live G.
import copy as _copy
G_ema = _copy.deepcopy(G).to(cfg.DEVICE)
G_ema.eval()
for p in G_ema.parameters():
    p.requires_grad_(False)

def update_ema(G_live, G_ema_model, decay):
    """In-place EMA update: ema_param = decay * ema_param + (1-decay) * live_param."""
    with torch.no_grad():
        for p_live, p_ema in zip(G_live.parameters(), G_ema_model.parameters()):
            p_ema.data.mul_(decay).add_(p_live.data, alpha=1.0 - decay)

print(f"EMA Generator initialised (enabled={cfg.USE_EMA}, decay={cfg.EMA_DECAY}, start_epoch={cfg.EMA_START_EPOCH})")

def active_generator():
    return G_ema if cfg.USE_EMA else G

def set_gan_lr(epoch):
    lr_g, lr_d = cfg.GAN_LR_G, cfg.GAN_LR_D
    for start_ep, stage_lr_g, stage_lr_d in cfg.GAN_LR_DECAY_EPOCHS:
        if epoch >= start_ep:
            lr_g, lr_d = stage_lr_g, stage_lr_d
    for group in optimizer_G.param_groups:
        group["lr"] = lr_g
    for group in optimizer_D.param_groups:
        group["lr"] = lr_d
    return lr_g, lr_d

# AMP scalers
try:
    from torch.amp import GradScaler as _GS2
    scaler_G = _GS2(device=cfg.DEVICE, enabled=use_amp)
    scaler_D = _GS2(device=cfg.DEVICE, enabled=use_amp)
except:
    from torch.cuda.amp import GradScaler as _GS2
    scaler_G = _GS2(enabled=use_amp); scaler_D = _GS2(enabled=use_amp)

# FID metric
from torchmetrics.image.fid import FrechetInceptionDistance
fid_metric = FrechetInceptionDistance(feature=cfg.FID_FEATURE).to(cfg.DEVICE)

# Tracking
G_losses, D_losses = [], []
mse_per_epoch  = []
fid_history    = []   # (epoch, fid)
wma_fid_history= []   # (epoch, wma_fid)
start_epoch    = 1
best_fid       = float("inf")
best_mse       = float("inf")
best_wma_fid   = float("inf")

# ── Resume ────────────────────────────────────────────────────────────────────
gan_latest = sorted(Path(cfg.CKPT_DIR).glob("gan_epoch*.pt"))
ck, gan_resume_path = None, None
if cfg.RESUME and gan_latest:
    for ck_path in reversed(gan_latest):
        ck = safe_torch_load(ck_path, cfg.DEVICE, "GAN resume")
        if ck is not None:
            gan_resume_path = ck_path
            break
if cfg.RESUME and ck is not None:
    G.load_state_dict(ck["G_state"])
    D.load_state_dict(ck["D_state"])
    optimizer_G.load_state_dict(ck["opt_G"])
    optimizer_D.load_state_dict(ck["opt_D"])
    start_epoch     = ck.get("epoch", 0) + 1
    best_fid        = ck.get("best_fid", float("inf"))
    best_mse        = ck.get("best_mse", float("inf"))
    best_wma_fid    = ck.get("best_wma_fid", float("inf"))
    G_losses        = ck.get("G_losses", [])
    D_losses        = ck.get("D_losses", [])
    fid_history     = ck.get("fid_history", [])
    wma_fid_history = ck.get("wma_fid_history", [])
    mse_per_epoch   = ck.get("mse_per_epoch", [])
    # Restore G_ema if saved in checkpoint
    if "G_ema_state" in ck:
        G_ema.load_state_dict(ck["G_ema_state"])
        print(f"[GAN RESUME] {gan_resume_path.name} | epoch={start_epoch}, best_fid={best_fid:.4f}, "
              f"best_wma_fid={best_wma_fid:.4f} | G_ema restored")
    else:
        # Warm-start EMA from G weights (no EMA state saved)
        G_ema.load_state_dict(G.state_dict())
        print(f"[GAN RESUME] {gan_resume_path.name} | epoch={start_epoch}, best_fid={best_fid:.4f}, "
              f"best_wma_fid={best_wma_fid:.4f} | G_ema warm-started from G")
else:
    print("[GAN FRESH] Training from scratch.")

print(f"Training ACFD-GAN | Grade {cfg.TARGET_CLASS} | "
      f"{cfg.GAN_EPOCHS} epochs | {len(dataloader)} steps/ep")
print("=" * 60)

In [ ]:
def mask_to_preview_rgb(masks):
    """Render semantic mask channels as paper-like RGB: vessel white, bright magenta, dark cyan, OD red."""
    masks = ((masks + 1.0) * 0.5).clamp(0, 1)
    if masks.size(1) < 4:
        return masks[:, :3]
    vessel, bright, dark, od = [masks[:, i:i+1] for i in range(4)]
    r = torch.clamp(vessel + bright + od, 0, 1)
    g = torch.clamp(vessel + dark, 0, 1)
    b = torch.clamp(vessel + bright + dark * 0.9, 0, 1)
    return torch.cat([r, g, b], dim=1)

def lesion_alpha_from_mask(masks, cfg):
    # masks in [-1, 1], channels: vessel, bright lesion, dark lesion, OD.
    if masks.size(1) < 3:
        return torch.zeros(masks.size(0), 1, masks.size(2), masks.size(3), device=masks.device, dtype=masks.dtype)
    mask01 = (masks + 1.0) * 0.5
    alpha = (mask01[:, 1:2] + mask01[:, 2:3]).clamp(0.0, 1.0)
    k = int(getattr(cfg, "INPAINT_ALPHA_BLUR", 1))
    if k > 1:
        if k % 2 == 0:
            k += 1
        alpha = F.avg_pool2d(alpha, kernel_size=k, stride=1, padding=k // 2)
    alpha = (alpha * float(getattr(cfg, "INPAINT_ALPHA_GAIN", 1.0))).clamp(0.0, float(getattr(cfg, "INPAINT_ALPHA_MAX", 1.0)))
    return alpha


def compose_lesion_inpaint(raw_fake, base_img, masks, cfg):
    if not getattr(cfg, "USE_LESION_INPAINT_GAN", False) or base_img is None:
        return raw_fake
    alpha = lesion_alpha_from_mask(masks, cfg)
    # Copy stable fundus anatomy from the base image; synthesize only lesion-heavy regions.
    return (base_img * (1.0 - alpha) + raw_fake * alpha).clamp(-1.0, 1.0)


def synthesize_fake(G, masks, z, cfg, base_img=None):
    raw_fake = G(masks, z)
    return compose_lesion_inpaint(raw_fake, base_img, masks, cfg)


def compute_wma_fid(fid_hist, window=5):
    """Weighted Moving Average FID (paper: window=5 or 7)."""
    n = min(len(fid_hist), window)
    if n == 0: return float("inf")
    recent = [f for _, f in fid_hist[-n:]]
    weights = list(range(1, n+1))
    return sum(w*f for w,f in zip(weights, recent)) / sum(weights)


def get_dataset_crop(dataset, idx):
    """Return the same 256x256 crop style used by __getitem__ from cached tensors."""
    support = dataset.supports[idx].float() if hasattr(dataset, "supports") else None
    img, mask = dataset._random_crop_pair(dataset.imgs[idx].float(), dataset.masks[idx].float(), support)
    return img, mask


def compute_fid_score(G, wae, dataset, cfg, LATENT_DIM):
    """Compute same-preprocess FID; optionally use best-of-K generated candidates."""
    fid_metric.reset()
    fid_n = len(dataset) if cfg.FID_SUBSET is None else min(cfg.FID_SUBSET, len(dataset))
    loader = DataLoader(dataset, batch_size=cfg.GEN_BATCH_FID,
                        shuffle=True, num_workers=0)
    collected = 0
    for batch_data in loader:
        imgs = batch_data[0]
        if collected >= fid_n: break
        take  = min(imgs.size(0), fid_n - collected)
        chunk = ((imgs[:take] + 1)*127.5).clamp(0,255).byte().to(cfg.DEVICE)
        fid_metric.update(chunk, real=True)
        collected += take

    generated_fid = 0
    while generated_fid < fid_n:
        b = min(cfg.GEN_BATCH_FID, fid_n - generated_fid)
        with torch.no_grad():
            # Pick random real 256x256 crops as conditioning.
            idx   = torch.randint(0, len(dataset), (b,))
            crop_pairs = [get_dataset_crop(dataset, int(idx_v)) for idx_v in idx]
            imgs_r  = torch.stack([p[0] for p in crop_pairs]).to(cfg.DEVICE)
            masks_b = torch.stack([p[1] for p in crop_pairs]).to(cfg.DEVICE)
            # Encode the same random crop for WAE latent.
            wae_z  = wae.encode(imgs_r)
            cand_n = max(1, int(getattr(cfg, "FID_CANDIDATES_PER_MASK", 1)))
            if cand_n == 1:
                noise = torch.randn(b, cfg.NOISE_DIM, device=cfg.DEVICE)
                z      = torch.cat([wae_z, noise], dim=1)
                fake   = synthesize_fake(G, masks_b, z, cfg, base_img=imgs_r)
            else:
                real_norm = (imgs_r.float() * 0.5 + 0.5 - vgg_loss.mean) / vgg_loss.std
                real_feat = vgg_loss.features(real_norm)
                best_score = torch.full((b,), float("inf"), device=cfg.DEVICE)
                best_fake = None
                cand_chunk = max(1, int(getattr(cfg, "FID_CANDIDATE_CHUNK", 1)))
                done = 0
                while done < cand_n:
                    k = min(cand_chunk, cand_n - done)
                    masks_rep = masks_b.repeat_interleave(k, dim=0)
                    wae_rep   = wae_z.repeat_interleave(k, dim=0)
                    noise = torch.randn(b * k, cfg.NOISE_DIM, device=cfg.DEVICE)
                    z     = torch.cat([wae_rep, noise], dim=1)
                    cand  = synthesize_fake(G, masks_rep, z, cfg, base_img=imgs_r.repeat_interleave(k, dim=0))
                    cand_norm = (cand.float() * 0.5 + 0.5 - vgg_loss.mean) / vgg_loss.std
                    cand_feat = vgg_loss.features(cand_norm)
                    _, c, h, w = real_feat.shape
                    scores = (cand_feat.view(b, k, c, h, w) - real_feat[:, None]).abs().mean(dim=(2,3,4))
                    chunk_score, chunk_idx = scores.min(dim=1)
                    cand_view = cand.view(b, k, *cand.shape[1:])
                    chunk_fake = cand_view[torch.arange(b, device=cfg.DEVICE), chunk_idx]
                    improve = chunk_score < best_score
                    if best_fake is None:
                        best_fake = chunk_fake.detach().clone()
                        best_score = chunk_score
                    elif improve.any():
                        best_fake[improve] = chunk_fake[improve].detach()
                        best_score[improve] = chunk_score[improve]
                    del cand, cand_feat, cand_norm, scores, chunk_fake
                    done += k
                fake = best_fake
            fake_u = ((fake + 1)*127.5).clamp(0,255).byte()
            fid_metric.update(fake_u, real=False)
        generated_fid += b
    return fid_metric.compute().item()


def compute_fid_multiseed(G, wae, dataset, cfg, LATENT_DIM, seeds=None):
    """Evaluate standard FID over fixed seeds and return mean/std/values."""
    seeds = list(cfg.POSTHOC_FID_SEEDS if seeds is None else seeds)
    if not seeds:
        raise ValueError("compute_fid_multiseed requires at least one seed")

    py_state = random.getstate()
    np_state = np.random.get_state()
    cpu_state = torch.random.get_rng_state()
    cuda_states = torch.cuda.get_rng_state_all() if cfg.DEVICE == "cuda" else None
    vals = []
    try:
        for seed in seeds:
            random.seed(seed)
            np.random.seed(seed)
            torch.manual_seed(seed)
            if cfg.DEVICE == "cuda":
                torch.cuda.manual_seed_all(seed)
            vals.append(compute_fid_score(G, wae, dataset, cfg, LATENT_DIM))
    finally:
        random.setstate(py_state)
        np.random.set_state(np_state)
        torch.random.set_rng_state(cpu_state)
        if cuda_states is not None:
            torch.cuda.set_rng_state_all(cuda_states)

    vals_np = np.asarray(vals, dtype=np.float32)
    return float(vals_np.mean()), float(vals_np.std()), vals



# ── Optional VGG feature cache ────────────────────────────────────────────────
# Safe for the resize-256 baseline. Keep disabled if crop_source_size > img_size.
if cfg.CACHE_VGG_FEATURES and dataset.cache_size == dataset.size and not hasattr(dataset, "vgg_feats"):
    print("Caching VGG features for real images...")
    vgg_loss.eval()
    vgg_feats_tmp = [[], [], []]
    with torch.no_grad():
        for i in tqdm(range(len(dataset)), desc="VGG Cache"):
            img = dataset.imgs[i].float().unsqueeze(0).to(cfg.DEVICE)
            yn = (img * 0.5 + 0.5 - vgg_loss.mean) / vgg_loss.std
            feats = vgg_loss.extract_features(yn)
            for j, feat in enumerate(feats):
                vgg_feats_tmp[j].append(feat.squeeze(0).cpu().half())
    dataset.vgg_feats = tuple(torch.stack(v) for v in vgg_feats_tmp)
    dataset.use_vgg_cache = True
gan_wall_start = time.perf_counter()

for epoch in range(start_epoch, cfg.GAN_EPOCHS + 1):
    lr_g_cur, lr_d_cur = set_gan_lr(epoch)
    if (time.perf_counter() - gan_wall_start) > cfg.MAX_TRAIN_SECONDS * 0.50:
        print(f"[TIME GUARD] GAN stopped at epoch {epoch}"); break

    G.train(); D.train()
    ep_g = ep_d = ep_l1 = ep_mse = 0.0
    n_batch = 0

    for batch_data in dataloader:
        if len(batch_data) == 4:
            imgs, masks, labels, real_vgg_feats = batch_data
            real_vgg_feats = tuple(f.to(cfg.DEVICE, non_blocking=True).float() for f in real_vgg_feats)
        else:
            imgs, masks, labels = batch_data
            real_vgg_feats = None
        imgs  = imgs.to(cfg.DEVICE, non_blocking=True)
        masks = masks.to(cfg.DEVICE, non_blocking=True)
        B     = imgs.size(0)
        real_label = torch.ones (B, device=cfg.DEVICE)
        fake_label = torch.zeros(B, device=cfg.DEVICE)

        # ── WAE latent (frozen encoder) ──────────────────────────────────
        with torch.no_grad():
            wae_z = wae.encode(imgs)                   # (B, 128)
        noise = torch.randn(B, cfg.NOISE_DIM,
                            device=cfg.DEVICE)          # (B, 128)
        z     = torch.cat([wae_z, noise], dim=1)       # (B, 256)

        # ── Train Discriminator ──────────────────────────────────────────
        global_step = (epoch - 1) * len(dataloader) + n_batch
        apply_r1 = (cfg.R1_GAMMA > 0 and global_step % cfg.R1_INTERVAL == 0)
        real_for_d = imgs.detach().requires_grad_(apply_r1)
        optimizer_D.zero_grad(set_to_none=True)
        with autocast(use_amp):
            with torch.no_grad():
                fake_imgs = synthesize_fake(G, masks, z, cfg, base_img=imgs)
            out_real = D(real_for_d, masks).view(B, -1).mean(1)
            out_fake = D(fake_imgs, masks).view(B, -1).mean(1)
            errD = (criterion_adv(out_real, real_label) +
                    criterion_adv(out_fake, fake_label)) * 0.5
        if apply_r1:
            grad_real = torch.autograd.grad(
                outputs=out_real.sum(), inputs=real_for_d,
                create_graph=True, retain_graph=True, only_inputs=True)[0]
            r1_penalty = grad_real.float().pow(2).flatten(1).sum(1).mean()
            errD = errD + 0.5 * cfg.R1_GAMMA * r1_penalty
        if not torch.isfinite(errD):
            raise FloatingPointError(
                f"Non-finite D loss at epoch={epoch}, batch={n_batch}, step={global_step}")
        scaler_D.scale(errD).backward()
        scaler_D.step(optimizer_D); scaler_D.update()

        # ── Train Generator ──────────────────────────────────────────────
        optimizer_G.zero_grad(set_to_none=True)
        with autocast(use_amp):
            fake_imgs = synthesize_fake(G, masks, z, cfg, base_img=imgs)
            out_fake  = D(fake_imgs, masks).view(B, -1).mean(1)
            errG_adv  = criterion_adv(out_fake, real_label)
            errG_l1   = F.l1_loss(fake_imgs, imgs)
            errG_vgg  = vgg_loss(fake_imgs, real=imgs, real_feat_cached=real_vgg_feats)
            if getattr(cfg, "USE_LESION_INPAINT_GAN", False):
                bg_w = 1.0 - lesion_alpha_from_mask(masks, cfg)
                errG_bg = (bg_w * (fake_imgs - imgs).abs()).mean()
            else:
                errG_bg = fake_imgs.new_tensor(0.0)
            errG = (errG_adv + cfg.LAMBDA_L1 * errG_l1 + cfg.LAMBDA_VGG * errG_vgg +
                    getattr(cfg, "BACKGROUND_PRESERVE_WEIGHT", 0.0) * errG_bg)
        if not torch.isfinite(errG):
            raise FloatingPointError(
                f"Non-finite G loss at epoch={epoch}, batch={n_batch}, step={global_step}")
        scaler_G.scale(errG).backward()
        scaler_G.unscale_(optimizer_G)
        torch.nn.utils.clip_grad_norm_(G.parameters(), max_norm=cfg.GRAD_CLIP_G_MAX_NORM)
        scaler_G.step(optimizer_G); scaler_G.update()

        # EMA update after each generator step
        if cfg.USE_EMA and epoch >= cfg.EMA_START_EPOCH:
            update_ema(G, G_ema, cfg.EMA_DECAY)

        ep_g   += errG.item()
        ep_d   += errD.item()
        ep_l1  += errG_l1.item()
        ep_mse += F.mse_loss(fake_imgs.detach(), imgs).item()
        n_batch += 1

    mean_g   = ep_g   / max(n_batch, 1)
    mean_d   = ep_d   / max(n_batch, 1)
    mean_l1  = ep_l1  / max(n_batch, 1)
    mean_mse = ep_mse / max(n_batch, 1)
    G_losses.append(mean_g)
    D_losses.append(mean_d)
    mse_per_epoch.append(mean_mse)

    # ── Logging ────────────────────────────────────────────────────────
    should_log = (epoch == start_epoch or epoch == cfg.GAN_EPOCHS or
                  epoch % cfg.LOG_EVERY == 0 or epoch % cfg.EVAL_FREQ == 0)
    if should_log:
        print(f"Epoch {epoch:3d}/{cfg.GAN_EPOCHS} | "
              f"G:{mean_g:.4f} D:{mean_d:.4f} | LR_G:{lr_g_cur:.1e} LR_D:{lr_d_cur:.1e} | "
              f"L1:{mean_l1:.4f} MSE:{mean_mse:.4f}")

    # ── Save only the important lightweight model states ────────────────
    if cfg.SAVE_BEST_MSE and mean_mse < best_mse:
        best_mse = mean_mse
        torch.save(G.state_dict(), Path(cfg.CKPT_DIR) / "best_generator_mse.pt")
        if should_log:
            print(f"  --> MSE improved! best={best_mse:.6f} [SAVED]")

    if cfg.SAVE_EVERY and epoch % cfg.SAVE_EVERY == 0:
        ck_path = Path(cfg.CKPT_DIR) / f"gan_epoch{epoch:04d}.pt"
        torch.save({"epoch":epoch, "G_state":G.state_dict(),
                    "best_fid":best_fid,
                    "best_mse":best_mse}, ck_path)
        print(f"  [CKPT] Saved: {ck_path.name}")

    # ── FID + WMA-FID evaluation ───────────────────────────────────────
    if epoch % cfg.EVAL_FREQ == 0 or epoch == cfg.GAN_EPOCHS:
        G.eval()
        G_eval = active_generator()
        G_eval.eval()
        with torch.no_grad():
            # Baseline uses live G; EMA can be re-enabled later as a separate ablation.
            current_fid, current_fid_std, current_fid_vals = compute_fid_multiseed(
                G_eval, wae, dataset, cfg, LATENT_DIM)
        fid_history.append((epoch, current_fid))
        wma_fid = compute_wma_fid(fid_history, cfg.WMA_WINDOW)
        wma_fid_history.append((epoch, wma_fid))
        fid_vals_txt = [round(float(v), 4) for v in current_fid_vals]
        print(f"  FID={current_fid:.4f}+/-{current_fid_std:.4f} | "
              f"WMA-FID(w={cfg.WMA_WINDOW})={wma_fid:.4f} | values={fid_vals_txt}")

        # Multi-seed mean-FID model selection: less sensitive to one lucky sample draw.
        if cfg.SAVE_BEST_FID and current_fid < best_fid:
            best_fid = current_fid
            # Save the exact generator used for FID selection.
            torch.save(G_eval.state_dict(),
                       Path(cfg.CKPT_DIR) / "best_generator_fid.pt")
            torch.save(G.state_dict(),
                       Path(cfg.CKPT_DIR) / "best_generator_fid_raw.pt")
            print(f"  --> FID improved! best={best_fid:.4f} [G_eval SAVED]")

        # WMA-FID model selection: paper-style, more robust to noisy epoch FID.
        if wma_fid < best_wma_fid:
            best_wma_fid = wma_fid
            if cfg.SAVE_WMA_MODEL:
                torch.save(G.state_dict(),
                           Path(cfg.CKPT_DIR) / "best_generator_wmafid.pt")
                print(f"  --> WMA-FID improved! best={best_wma_fid:.4f} [SAVED]")
            else:
                print(f"  --> WMA-FID improved! best={best_wma_fid:.4f}")

        # ── Preview images ───────────────────────────────────────────────
        with torch.no_grad():
            n_show = min(8, len(dataset))
            idx_v  = random.sample(range(len(dataset)), n_show)
            preview_pairs = [get_dataset_crop(dataset, v) for v in idx_v]
            imgs_v  = torch.stack([p[0] for p in preview_pairs]).to(cfg.DEVICE)
            masks_v = torch.stack([p[1] for p in preview_pairs]).to(cfg.DEVICE)
            wae_zv  = wae.encode(imgs_v)
            nv      = torch.randn(n_show, cfg.NOISE_DIM, device=cfg.DEVICE)
            zv      = torch.cat([wae_zv, nv], dim=1)
            fake_v  = synthesize_fake(G_eval, masks_v, zv, cfg, base_img=imgs_v)

        grid_real = vutils.make_grid(
            ((imgs_v+1)/2).clamp(0,1), nrow=n_show, padding=2)
        grid_fake = vutils.make_grid(
            ((fake_v+1)/2).clamp(0,1), nrow=n_show, padding=2)
        grid_mask = vutils.make_grid(
            mask_to_preview_rgb(masks_v), nrow=n_show, padding=2)

        fig, axes = plt.subplots(3,1, figsize=(n_show*2, 7))
        for ax, g, lbl in zip(axes,
            [grid_real, grid_mask, grid_fake],
            ["Real","Mask","Generated"]):
            ax.imshow(g.permute(1,2,0).cpu().numpy())
            ax.axis("off"); ax.set_title(lbl, fontsize=10)
        plt.suptitle(
            f"ACFD-GAN Epoch {epoch}/{cfg.GAN_EPOCHS} | "
            f"FID={current_fid:.2f} | WMA-FID={wma_fid:.2f} | "
            f"Grade {cfg.TARGET_CLASS} ({GRADE_NAMES.get(cfg.TARGET_CLASS,'?')})",
            fontsize=11)
        plt.tight_layout()
        if cfg.SAVE_PREVIEWS:
            preview_path = Path(cfg.CKPT_DIR)/f"preview_ep{epoch:04d}.png"
            plt.savefig(preview_path, dpi=100, bbox_inches="tight")
        if epoch % cfg.SHOW_PREVIEW_EVERY == 0 or epoch == cfg.GAN_EPOCHS:
            plt.show()
        plt.close()

        G.train()
        if cfg.DEVICE == "cuda":
            torch.cuda.empty_cache()

active_generator().eval()
print(f"\nGAN Training done. Best FID={best_fid:.4f} | Best WMA-FID={best_wma_fid:.4f}")


## 8. Metrics Dashboard

Paper metrics: FID (per grade), WMA-FID, MSE, G/D losses.


In [ ]:
# ── Loss Curves ───────────────────────────────────────────────────────────────
if G_losses:
    fig, axes = plt.subplots(1, 3, figsize=(18, 4))

    ep_range = range(1, len(G_losses)+1)
    w = 5

    # G/D losses
    axes[0].plot(ep_range, G_losses, color="#3498db", lw=1, alpha=0.6, label="G loss")
    axes[0].plot(ep_range, D_losses, color="#e74c3c", lw=1, alpha=0.6, label="D loss")
    g_sm = pd.Series(G_losses).rolling(w, min_periods=1).mean()
    d_sm = pd.Series(D_losses).rolling(w, min_periods=1).mean()
    axes[0].plot(ep_range, g_sm, "#2980b9", lw=2)
    axes[0].plot(ep_range, d_sm, "#c0392b", lw=2)
    axes[0].set_title("G / D Losses (ACFD-GAN)"); axes[0].legend()
    axes[0].set_xlabel("Epoch"); axes[0].grid(alpha=0.3)

    # FID & WMA-FID
    if fid_history:
        fid_ep  = [ep for ep,_ in fid_history]
        fid_val = [f  for _,f  in fid_history]
        wma_ep  = [ep for ep,_ in wma_fid_history]
        wma_val = [f  for _,f  in wma_fid_history]
        axes[1].plot(fid_ep, fid_val, "o-", color="#9b59b6", lw=1.5,
                     markersize=4, label="FID")
        axes[1].plot(wma_ep, wma_val, "s--", color="#e67e22", lw=2,
                     markersize=5, label=f"WMA-FID (w={cfg.WMA_WINDOW})")
        axes[1].set_title("FID & WMA-FID per Epoch")
        axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("FID")
        axes[1].legend(); axes[1].grid(alpha=0.3)

    # MSE per epoch
    if mse_per_epoch:
        axes[2].plot(ep_range, mse_per_epoch, color="#27ae60", lw=1.5)
        axes[2].set_title(f"MSE (train) — Grade {cfg.TARGET_CLASS}")
        axes[2].set_xlabel("Epoch"); axes[2].set_ylabel("MSE")
        axes[2].grid(alpha=0.3)

    grade_name = GRADE_NAMES.get(cfg.TARGET_CLASS, "All")
    plt.suptitle(f"ACFD-GAN Training — Grade {cfg.TARGET_CLASS} ({grade_name})",
                 fontsize=13)
    plt.tight_layout()
    plt.savefig(f"{cfg.LOG_DIR}/acfdgan_metrics_{RUN_TAG}.png", dpi=150)
    plt.show()

    # ── Metrics Summary Table ─────────────────────────────────────────────
    print("\n" + "="*60)
    print(f"  ACFD-GAN METRICS SUMMARY — Grade {cfg.TARGET_CLASS} ({grade_name})")
    print("="*60)
    print(f"  Final G Loss      : {G_losses[-1]:.4f}")
    print(f"  Final D Loss      : {D_losses[-1]:.4f}")
    if fid_history:
        best_fid_val = min(f for _,f in fid_history)
        best_fid_ep  = fid_history[[f for _,f in fid_history].index(best_fid_val)][0]
        print(f"  Best FID          : {best_fid_val:.4f}  (epoch {best_fid_ep})")
        print(f"  Final FID         : {fid_history[-1][1]:.4f}")
    if wma_fid_history:
        print(f"  Best WMA-FID      : {best_wma_fid:.4f}")
        print(f"  Final WMA-FID     : {wma_fid_history[-1][1]:.4f}")
    if mse_per_epoch:
        print(f"  Final MSE (train) : {mse_per_epoch[-1]:.6f}")
        print(f"  Best MSE (train)  : {min(mse_per_epoch):.6f}")
    print("="*60)
else:
    print("No training history available.")


## 9. Generate Synthetic Images


In [ ]:
# Load best generator. Prefer raw FID for generation because the target is FID < 20;
# keep WMA-FID as the paper-style fallback.
best_fid_path = Path(cfg.CKPT_DIR) / "best_generator_fid.pt"
best_mse_path = Path(cfg.CKPT_DIR) / "best_generator_mse.pt"
best_wma_path = Path(cfg.CKPT_DIR) / "best_generator_wmafid.pt"
best_g_path = best_fid_path if best_fid_path.exists() else (best_mse_path if best_mse_path.exists() else best_wma_path)
best_g_ck = safe_torch_load(best_g_path, cfg.DEVICE, "best generator") if best_g_path.exists() else None
if best_g_ck is not None:
    G.load_state_dict(best_g_ck)
    if cfg.USE_EMA:
        G_ema.load_state_dict(best_g_ck)
    print(f"[Gen] Loaded generator: {best_g_path.name}")
else:
    ckpts = sorted(Path(cfg.CKPT_DIR).glob("gan_epoch*.pt"))
    ck = None
    fallback_path = None
    for ck_path in reversed(ckpts):
        ck = safe_torch_load(ck_path, cfg.DEVICE, "fallback generator")
        if ck is not None:
            fallback_path = ck_path
            break
    if ck is not None:
        G.load_state_dict(ck["G_state"])
        print(f"[Gen] Fallback checkpoint: {fallback_path.name}")
    else:
        raise FileNotFoundError("No trained GAN checkpoint found.")

G_gen = active_generator()
G_gen.eval(); wae.eval()
torch.cuda.empty_cache() if cfg.DEVICE == "cuda" else None

current_counts = df_full["diagnosis"].value_counts().sort_index().to_dict()
generated_records = []

print("\n--- Generation Plan ---")
for g in ACTIVE_CLASSES:
    need = max(0, cfg.TARGET_PER_CLASS - current_counts.get(g, 0))
    if cfg.MAX_GENERATE_PER_CLASS is not None:
        need = min(need, cfg.MAX_GENERATE_PER_CLASS)
    print(f"  Grade {g} ({GRADE_NAMES[g]:<13}): "
          f"{current_counts.get(g,0):4d} real -> generate {need:4d}")

print("\n--- Generating ---")
with torch.no_grad():
    for class_id in ACTIVE_CLASSES:
        cur   = current_counts.get(class_id, 0)
        need  = max(0, cfg.TARGET_PER_CLASS - cur)
        if cfg.MAX_GENERATE_PER_CLASS is not None:
            need = min(need, cfg.MAX_GENERATE_PER_CLASS)
        if need == 0:
            print(f"  Grade {class_id}: balanced, skip."); continue

        out_dir = Path(cfg.OUTPUT_DIR) / str(class_id)
        out_dir.mkdir(parents=True, exist_ok=True)

        # Collect masks from real images of this class for conditioning
        cls_df  = df_full[df_full["diagnosis"] == class_id]
        gen_cnt = 0
        pbar    = tqdm(total=need, desc=f"  Grade {class_id}", leave=False)

        while gen_cnt < need:
            bs = min(cfg.GEN_BATCH, need - gen_cnt)
            # Sample random real images from this class for mask+WAE latent
            sampled = cls_df.sample(bs, replace=(len(cls_df) < bs))
            masks_b_list, imgs_b_list = [], []
            for _, row in sampled.iterrows():
                p = os.path.join(cfg.IMG_DIR, f"{row['id_code']}.png")
                bg = cv2.imread(p)
                img_t, mask_t = preprocess_bgr_to_crop_tensors(
                    bg, cfg.IMG_SIZE, cfg.CROP_SOURCE_SIZE,
                    cfg.CLAHE_CLIP, cfg.CLAHE_TILE, random_crop=True,
                    min_fundus_frac=cfg.CROP_MIN_FUNDUS_FRAC)
                masks_b_list.append(mask_t)
                imgs_b_list.append(img_t)
            masks_b = torch.stack(masks_b_list).to(cfg.DEVICE)
            imgs_b  = torch.stack(imgs_b_list).to(cfg.DEVICE)

            # WAE latent + candidate noise filtering.
            wae_z = wae.encode(imgs_b.float())
            cand_n = max(1, cfg.GEN_CANDIDATES_PER_MASK)
            if cand_n == 1:
                noise = torch.randn(bs, cfg.NOISE_DIM, device=cfg.DEVICE)
                z     = torch.cat([wae_z, noise], dim=1)
                fakes = synthesize_fake(G_gen, masks_b, z, cfg, base_img=imgs_b)
            else:
                masks_rep = masks_b.repeat_interleave(cand_n, dim=0)
                wae_rep   = wae_z.repeat_interleave(cand_n, dim=0)
                noise = torch.randn(bs * cand_n, cfg.NOISE_DIM, device=cfg.DEVICE)
                z     = torch.cat([wae_rep, noise], dim=1)
                cand  = synthesize_fake(G_gen, masks_rep, z, cfg, base_img=imgs_b.repeat_interleave(cand_n, dim=0))
                with torch.no_grad():
                    real_feat = vgg_loss.features((imgs_b.float() * 0.5 + 0.5 - vgg_loss.mean) / vgg_loss.std)
                    cand_feat = vgg_loss.features((cand.float() * 0.5 + 0.5 - vgg_loss.mean) / vgg_loss.std)
                    _, c, h, w = real_feat.shape
                    scores = (cand_feat.view(bs, cand_n, c, h, w) - real_feat[:, None]).abs().mean(dim=(2,3,4))
                    best_idx = scores.argmin(dim=1)
                    fakes = cand.view(bs, cand_n, *cand.shape[1:])[torch.arange(bs, device=cfg.DEVICE), best_idx]

            for i in range(bs):
                fname = f"syn_g{class_id}_{gen_cnt+i:05d}.png"
                spath = str(out_dir / fname)
                cv2.imwrite(spath, tensor_to_bgr(fakes[i]))
                generated_records.append({
                    "id_code": f"syn_g{class_id}_{gen_cnt+i:05d}",
                    "diagnosis": class_id, "source": "synthetic",
                    "filepath": spath
                })
            gen_cnt += bs
            pbar.update(bs)
        pbar.close()
        print(f"  Grade {class_id} ({GRADE_NAMES[class_id]}): {gen_cnt} images generated")

print(f"\nTotal generated: {len(generated_records):,}")

# ── Post-hoc FID on saved synthetic images ──────────────────────────────────
def compute_posthoc_fid_for_class(class_id, max_real=None, max_fake=None, seed=None):
    metric = FrechetInceptionDistance(feature=cfg.FID_FEATURE).to(cfg.DEVICE)
    cls_df = df_full[df_full["diagnosis"] == class_id].reset_index(drop=True)
    if max_real is not None:
        cls_df = cls_df.sample(min(max_real, len(cls_df)), random_state=seed or cfg.SEED)

    real_tensors = []
    for _, row in cls_df.iterrows():
        bg = cv2.imread(os.path.join(cfg.IMG_DIR, f"{row['id_code']}.png"))
        img_t, _ = preprocess_bgr_to_crop_tensors(
            bg, cfg.IMG_SIZE, cfg.CROP_SOURCE_SIZE,
            cfg.CLAHE_CLIP, cfg.CLAHE_TILE, random_crop=False,
            min_fundus_frac=cfg.CROP_MIN_FUNDUS_FRAC)
        real_tensors.append(((img_t + 1) * 127.5).clamp(0,255).byte())

    fake_paths = sorted((Path(cfg.OUTPUT_DIR) / str(class_id)).glob("*.png"))
    rng = np.random.default_rng(seed if seed is not None else cfg.SEED)
    if max_fake is not None and len(fake_paths) > max_fake:
        fake_paths = [fake_paths[i] for i in rng.choice(len(fake_paths), size=max_fake, replace=False)]
    fake_tensors = []
    for p in fake_paths:
        bg = cv2.imread(str(p))
        if bg is None:
            continue
        rgb = cv2.cvtColor(cv2.resize(bg, (cfg.IMG_SIZE, cfg.IMG_SIZE)), cv2.COLOR_BGR2RGB)
        fake_tensors.append(torch.from_numpy(rgb.transpose(2,0,1)).byte())

    n = min(len(real_tensors), len(fake_tensors))
    if n < 2:
        return float("nan"), len(real_tensors), len(fake_tensors)
    real_tensors = real_tensors[:n]
    if len(fake_tensors) > n:
        keep = rng.choice(len(fake_tensors), size=n, replace=False)
        fake_tensors = [fake_tensors[i] for i in keep]
    else:
        fake_tensors = fake_tensors[:n]
    for i in range(0, n, cfg.GEN_BATCH_FID):
        metric.update(torch.stack(real_tensors[i:i+cfg.GEN_BATCH_FID]).to(cfg.DEVICE), real=True)
        metric.update(torch.stack(fake_tensors[i:i+cfg.GEN_BATCH_FID]).to(cfg.DEVICE), real=False)
    return metric.compute().item(), len(real_tensors), len(fake_tensors)

posthoc_fid_results = {}
for g in ACTIVE_CLASSES:
    vals = []
    n_real = n_fake = 0
    for seed in cfg.POSTHOC_FID_SEEDS:
        post_fid, n_real, n_fake = compute_posthoc_fid_for_class(g, seed=seed)
        vals.append(post_fid)
    vals_np = np.array(vals, dtype=np.float32)
    posthoc_fid_results[g] = vals
    print(f"[Post-hoc FID] Grade {g}: mean={np.nanmean(vals_np):.4f} std={np.nanstd(vals_np):.4f} "
          f"values={[round(float(v),4) for v in vals]} (n={min(n_real,n_fake)}, real={n_real}, fake={n_fake})")

# Preview
n_active = len(ACTIVE_CLASSES)
fig, axes = plt.subplots(n_active, 5, figsize=(13, n_active*2.5+0.5),
                         squeeze=False)
with torch.no_grad():
    for ri, g in enumerate(ACTIVE_CLASSES):
        cls_df_g = df_full[df_full["diagnosis"] == g]
        sampled  = cls_df_g.sample(5, replace=(len(cls_df_g)<5))
        masks_l, imgs_l = [], []
        for _, row in sampled.iterrows():
            bg = cv2.imread(os.path.join(cfg.IMG_DIR, f"{row['id_code']}.png"))
            img_t, mask_t = preprocess_bgr_to_crop_tensors(
                bg, cfg.IMG_SIZE, cfg.CROP_SOURCE_SIZE,
                cfg.CLAHE_CLIP, cfg.CLAHE_TILE, random_crop=True,
                min_fundus_frac=cfg.CROP_MIN_FUNDUS_FRAC)
            masks_l.append(mask_t)
            imgs_l.append(img_t)
        ms = torch.stack(masks_l).to(cfg.DEVICE)
        im = torch.stack(imgs_l).to(cfg.DEVICE)
        wz = wae.encode(im)
        nz = torch.randn(5, cfg.NOISE_DIM, device=cfg.DEVICE)
        zz = torch.cat([wz, nz], dim=1)
        fk = G_gen(ms, zz).cpu()
        for j in range(5):
            ax = axes[ri, j]
            img = ((fk[j].permute(1,2,0).numpy()+1)/2).clip(0,1)
            ax.imshow(img); ax.axis("off")
            if j == 0:
                ax.set_ylabel(f"G{g} {GRADE_NAMES[g]}",
                              fontsize=8, rotation=0, ha="right", va="center")
plt.suptitle("ACFD-GAN Generated Samples (5 per grade)", fontsize=12)
plt.tight_layout()
plt.savefig(f"{cfg.LOG_DIR}/acfdgan_generated_preview_{RUN_TAG}.png",
            dpi=120, bbox_inches="tight")
plt.show()

## 10. Save Augmented CSV & Verify Distribution


In [ ]:
# ── Build augmented CSV ────────────────────────────────────────────────────────
real_part = df_full[["id_code","diagnosis"]].copy()
real_part["source"]   = "real"
real_part["filepath"] = real_part["id_code"].apply(
    lambda x: os.path.join(cfg.IMG_DIR, f"{x}.png"))

if len(generated_records) > 0:
    syn_part = pd.DataFrame(generated_records)
else:
    syn_part = pd.DataFrame(columns=["id_code","diagnosis","source","filepath"])
    print("[CSV] No synthetic records; CSV = real only.")

aug_df = pd.concat([real_part, syn_part], ignore_index=True)
aug_df.to_csv(cfg.AUG_CSV, index=False)
print(f"Augmented CSV saved: {cfg.AUG_CSV}")
print(f"Total: {len(aug_df):,} (real:{len(real_part):,} | syn:{len(syn_part):,})")

# ── Final distribution plot ───────────────────────────────────────────────────
final = aug_df["diagnosis"].value_counts().sort_index()
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

counts_real = df_full["diagnosis"].value_counts().sort_index()
axes[0].bar(counts_real.index, counts_real.values,
            color=[COLORS[i] for i in counts_real.index],
            edgecolor="black", alpha=0.85)
for i, (idx, v) in enumerate(counts_real.items()):
    axes[0].text(idx, v+20, str(v), ha="center", fontsize=9, fontweight="bold")
axes[0].set_title("Before Augmentation")
axes[0].set_xticks(range(5))
axes[0].set_xticklabels([f"G{i}\n{GRADE_NAMES[i]}" for i in range(5)], fontsize=8)
axes[0].set_ylabel("Count")

after_counts = [final.get(i,0) for i in range(5)]
axes[1].bar(range(5), after_counts, color=COLORS, edgecolor="black", alpha=0.85)
axes[1].axhline(cfg.TARGET_PER_CLASS, color="red", ls="--", lw=2,
                label=f"Target={cfg.TARGET_PER_CLASS}")
for i, v in enumerate(after_counts):
    axes[1].text(i, v+20, f"{v:,}", ha="center", fontsize=9, fontweight="bold")
axes[1].set_title("After ACFD-GAN Augmentation")
axes[1].set_xticks(range(5))
axes[1].set_xticklabels([f"G{i}\n{GRADE_NAMES[i]}" for i in range(5)], fontsize=8)
axes[1].set_ylabel("Count"); axes[1].legend()

plt.suptitle(f"APTOS 2019 — ACFD-GAN Augmentation (run={RUN_TAG})", fontsize=12)
plt.tight_layout()
plt.savefig(f"{cfg.LOG_DIR}/acfdgan_distribution_{RUN_TAG}.png", dpi=150)
plt.show()

print("\n--- Final Class Distribution ---")
for g in range(5):
    cnt = final.get(g, 0)
    bar = "X" * (cnt // 100)
    print(f"  Grade {g} ({GRADE_NAMES[g]:<13}): {cnt:5,}  {bar}")


## 11. Final Metrics Summary (Paper Comparison)


In [ ]:
print("\n" + "="*65)
print("  ACFD-GAN PAPER COMPARISON METRICS")
print("="*65)
print(f"  Grade         : {cfg.TARGET_CLASS} ({GRADE_NAMES.get(cfg.TARGET_CLASS,'All')})")
print(f"  WAE epochs    : {cfg.WAE_EPOCHS}")
print(f"  GAN epochs    : {cfg.GAN_EPOCHS}")
print(f"  Adv loss      : {cfg.ADV_LOSS_TYPE}")
print(f"  Loss weights  : Adv=1, L1={cfg.LAMBDA_L1}, VGG={cfg.LAMBDA_VGG}")
print(f"  WMA window    : {cfg.WMA_WINDOW}")
print("-"*65)

if G_losses:
    print(f"  Final G Loss    : {G_losses[-1]:.6f}")
    print(f"  Final D Loss    : {D_losses[-1]:.6f}")
    print(f"  Min   G Loss    : {min(G_losses):.6f}")

if mse_per_epoch:
    print(f"  Final MSE       : {mse_per_epoch[-1]:.6f}  (training pair MSE; not paper MSE)")
    print(f"  Best  MSE       : {min(mse_per_epoch):.6f}")

if fid_history:
    fid_vals = [f for _,f in fid_history]
    best_fid_v  = min(fid_vals)
    best_fid_ep = fid_history[fid_vals.index(best_fid_v)][0]
    print(f"  Best FID        : {best_fid_v:.4f}   (epoch {best_fid_ep})")
    print(f"  Final FID       : {fid_history[-1][1]:.4f}  (same-preprocess in-training FID)")

if wma_fid_history:
    wma_vals = [f for _,f in wma_fid_history]
    print(f"  Best WMA-FID    : {best_wma_fid:.4f}  (paper-style model selection)")
    print(f"  Final WMA-FID   : {wma_fid_history[-1][1]:.4f}")

print("-"*65)
print("  Paper Reference FID values (from Table 5):")
paper_fid = {0: 13.51, 1: 14.77, 2: 15.06, 3: 15.37, 4: 15.29}
paper_mse = {0: 0.0091, 1: 0.0107, 2: 0.0112, 3: 0.0116, 4: 0.0114}
for g in range(5):
    fid_ref = paper_fid.get(g, "N/A")
    mse_ref = paper_mse.get(g, "N/A")
    print(f"    Grade {g} ({GRADE_NAMES[g]:<13}): FID={fid_ref}  MSE={mse_ref}")
print("  * Note: paper values are approximate; verify against your paper copy.")
print("="*65)

# FID comparison bar chart
if fid_history:
    fig, ax = plt.subplots(figsize=(9, 4))
    fid_eps = [ep for ep,_ in fid_history]
    fid_vs  = [f  for _,f  in fid_history]
    ax.plot(fid_eps, fid_vs, "o-", color="#9b59b6", lw=1.5,
            markersize=4, label="FID (ours)")
    if wma_fid_history:
        wm_eps = [ep for ep,_ in wma_fid_history]
        wm_vs  = [f  for _,f  in wma_fid_history]
        ax.plot(wm_eps, wm_vs, "s--", color="#e67e22", lw=2,
                markersize=5, label=f"WMA-FID w={cfg.WMA_WINDOW}")
    if cfg.TARGET_CLASS in paper_fid:
        ax.axhline(paper_fid[cfg.TARGET_CLASS], color="red", ls=":",
                   lw=2, label=f"Paper FID={paper_fid[cfg.TARGET_CLASS]}")
    grade_name = GRADE_NAMES.get(cfg.TARGET_CLASS, "All")
    ax.set_title(f"FID History — Grade {cfg.TARGET_CLASS} ({grade_name})")
    ax.set_xlabel("Epoch"); ax.set_ylabel("FID")
    ax.legend(); ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(f"{cfg.LOG_DIR}/acfdgan_fid_comparison_{RUN_TAG}.png", dpi=120)
    plt.show()


## 12. Package Output as ZIP

Zip tất cả ảnh sinh ra + CSV + metrics plots để tải về từ Kaggle.


In [ ]:
import shutil, zipfile

if not getattr(cfg, "PACKAGE_OUTPUT", False):
    print("[ZIP] Skipped because cfg.PACKAGE_OUTPUT=False.")
    print("      This avoids duplicating generated images/checkpoints and filling Kaggle output.")
else:
    # ── 1. Zip generated images ────────────────────────────────────────────────
    img_zip_name = f"{cfg.LOG_DIR}/acfdgan_images_{RUN_TAG}"
    if os.path.exists(cfg.OUTPUT_DIR) and any(Path(cfg.OUTPUT_DIR).rglob("*.png")):
        shutil.make_archive(img_zip_name, "zip", cfg.OUTPUT_DIR)
        img_zip_size = os.path.getsize(f"{img_zip_name}.zip") / 1e6
        print(f"[ZIP] Images: {img_zip_name}.zip  ({img_zip_size:.1f} MB)")
    else:
        print("[ZIP] No generated images found.")

    # ── 2. Zip checkpoints (optional, for resume) ──────────────────────────────
    ckpt_zip_name = f"{cfg.LOG_DIR}/acfdgan_checkpoints_{RUN_TAG}"
    if os.path.exists(cfg.CKPT_DIR):
        shutil.make_archive(ckpt_zip_name, "zip", cfg.CKPT_DIR)
        ckpt_zip_size = os.path.getsize(f"{ckpt_zip_name}.zip") / 1e6
        print(f"[ZIP] Checkpoints: {ckpt_zip_name}.zip  ({ckpt_zip_size:.1f} MB)")

    # ── 3. Bundle: images + CSV + metric plots into 1 zip ─────────────────────
    bundle_zip = f"{cfg.LOG_DIR}/acfdgan_full_bundle_{RUN_TAG}.zip"
    with zipfile.ZipFile(bundle_zip, "w", zipfile.ZIP_DEFLATED) as zf:
        for img_f in Path(cfg.OUTPUT_DIR).rglob("*.png"):
            zf.write(img_f, arcname=f"generated/{img_f.parent.name}/{img_f.name}")
        if os.path.exists(cfg.AUG_CSV):
            zf.write(cfg.AUG_CSV, arcname=f"train_augmented_acfdgan_{RUN_TAG}.csv")
        for png in Path(cfg.LOG_DIR).glob("acfdgan_*.png"):
            zf.write(png, arcname=f"plots/{png.name}")
        for png in Path(cfg.LOG_DIR).glob("wae_*.png"):
            zf.write(png, arcname=f"plots/{png.name}")

    bundle_size = os.path.getsize(bundle_zip) / 1e6
    print(f"\n[BUNDLE] {bundle_zip}  ({bundle_size:.1f} MB)")
